Check also for asymmetree number of events compared to number of internal nodes in the parasite tree

# General functions for tree analysis

In [ ]:
#print number of multiple associations (symbiotns)
def count_multiple_associations(datasets):
    multiple_assoc_counts = []
    for dataset in datasets:
        assoc_dict = {}
        for parasite, host in dataset['associations']:
            if parasite not in assoc_dict:
                assoc_dict[parasite] = set()
            assoc_dict[parasite].add(host)
        multiple_count = sum(1 for hosts in assoc_dict.values() if len(hosts) > 1)
        multiple_assoc_counts.append(multiple_count)
    return multiple_assoc_counts

In [ ]:
# Tree length, and number of leaves in host and parasite trees
from Bio import Phylo
from io import StringIO
import re
import os
def load_tree(newick_str):

    return Phylo.read(StringIO(newick_str), "newick")

def get_tree_stats(tree):
    num_leaves = len(tree.get_terminals())
    total_branch_length = sum(clade.branch_length for clade in tree.get_nonterminals() + tree.get_terminals() if clade.branch_length)
    max_depth = max(tree.depths().values())
    num_branches = len(tree.get_nonterminals()) + len(tree.get_terminals()) - 1  # Total branches in the tree
    tree_stats = {
        "num_leaves": num_leaves,
        "total_branch_length": total_branch_length,
        "max_depth": max_depth,
        "num_branches": num_branches
    }
    return tree_stats

def host_tree_branches_list(datasets):
    values = []
    for dataset in datasets:
        if "host_tree_stats" in dataset:
            values.append(dataset["host_tree_stats"]["num_branches"])
    return values


def parasite_tree_branches_list(datasets):
    values = []
    for dataset in datasets:
        if "parasite_tree_stats" in dataset:
            values.append(dataset["parasite_tree_stats"]["num_branches"])
    return values


def host_tree_length_list(datasets):
    values = []
    for dataset in datasets:
        if "host_tree_stats" in dataset:
            values.append(dataset["host_tree_stats"]["max_depth"])
    return values


def parasite_tree_length_list(datasets):
    values = []
    for dataset in datasets:
        if "parasite_tree_stats" in dataset:
            values.append(dataset["parasite_tree_stats"]["max_depth"])
    return values


def host_tree_leaves_list(datasets):
    values = []
    for dataset in datasets:
        if "host_tree_stats" in dataset:
            values.append(dataset["host_tree_stats"]["num_leaves"])
    return values


def parasite_tree_leaves_list(datasets):
    values = []
    for dataset in datasets:
        if "parasite_tree_stats" in dataset:
            values.append(dataset["parasite_tree_stats"]["num_leaves"])
    return values

In [ ]:
def host_vs_parasite_list(datasets):
    differences = []
    for dataset in datasets:
        if 'host_tree_stats' in dataset and 'parasite_tree_stats' in dataset:
            host_depth = dataset['host_tree_stats']['num_branches']
            parasite_depth = dataset['parasite_tree_stats']['num_branches']
            differences.append(host_depth - parasite_depth)
    return differences

def host_vs_parasite_list_leaves(datasets):
    differences = []
    for dataset in datasets:
        if 'host_tree_stats' in dataset and 'parasite_tree_stats' in dataset:
            host_depth = dataset['host_tree_stats']['num_leaves']
            parasite_depth = dataset['parasite_tree_stats']['num_leaves']
            differences.append(host_depth - parasite_depth)
    return differences
    
def host_vs_parasite_list_length(datasets):
    differences = []
    for dataset in datasets:
        if 'host_tree_stats' in dataset and 'parasite_tree_stats' in dataset:
            host_depth = dataset['host_tree_stats']['max_depth']
            parasite_depth = dataset['parasite_tree_stats']['max_depth']
            differences.append(host_depth - parasite_depth)
    return differences

In [ ]:
def average_host_tree_depth(datasets): #num branches
    total_depth = 0
    count = 0
    for dataset in datasets:
        if 'host_tree_stats' in dataset:
            total_depth += dataset['host_tree_stats']['num_branches']
            count += 1
    return total_depth / count if count > 0 else 0


def average_parasite_tree_depth(datasets):#num branches
    total_depth = 0
    count = 0
    for dataset in datasets:
        if 'parasite_tree_stats' in dataset:
            total_depth += dataset['parasite_tree_stats']['num_branches']
            count += 1
    return total_depth / count if count > 0 else 0



def average_host_tree_length(datasets): # max depth
    total_length = 0
    count = 0
    for dataset in datasets:
        if 'host_tree_stats' in dataset:
            total_length += dataset['host_tree_stats']['max_depth']
            count += 1
    return total_length / count if count > 0 else 0


def average_parasite_tree_length(datasets):# max depth
    total_length = 0
    count = 0
    for dataset in datasets:
        if 'parasite_tree_stats' in dataset:
            total_length += dataset['parasite_tree_stats']['max_depth']
            count += 1
    return total_length / count if count > 0 else 0


def average_host_tree_leaves(datasets):
    total_leaves = 0
    count = 0
    for dataset in datasets:
        if 'host_tree_stats' in dataset:
            total_leaves += dataset['host_tree_stats']['num_leaves']
            count += 1
    return total_leaves / count if count > 0 else 0

def average_parasite_tree_leaves(datasets):
    total_leaves = 0
    count = 0
    for dataset in datasets:
        if 'parasite_tree_stats' in dataset:
            total_leaves += dataset['parasite_tree_stats']['num_leaves']
            count += 1
    return total_leaves / count if count > 0 else 0 

In [ ]:
# extract associations and count the number of associations per dataset. Measure the average
def average_associations_per_dataset(datasets):
    total_associations = 0
    count = 0
    for dataset in datasets:
        if 'associations' in dataset:
            total_associations += len(dataset['associations'])
            count += 1
    return total_associations / count if count > 0 else 0

In [ ]:
# phylogenetic tree metrics, such as cherry index, Sackin index, and Colless' index
import pandas as pd
from ete3 import Tree
import matplotlib.pyplot as plt
import seaborn as sns

def count_cherries(tree):
    count = 0
    for node in tree.traverse():
        if not node.is_leaf():
            children = node.get_children()
            if len(children) == 2 and all(child.is_leaf() for child in children):
                count += 1
    return count

def cherry_index(tree):
    cherries = count_cherries(tree)
    leaves = len(tree.get_leaves())
    cherry_index = (cherries / leaves if leaves > 0 else 0)*2
    return cherry_index
 

def sackin_index(tree):
    sacking_index = sum(tree.get_distance(leaf) for leaf in tree.iter_leaves())
    leaves = len(tree.get_leaves())
    sacking_index = sacking_index / (leaves**2) if leaves > 0 else 0
    return sacking_index

def sackin_index_topology(tree):
    sacking_index = sum(tree.get_distance(leaf,topology_only = True) for leaf in tree.iter_leaves())
    leaves = len(tree.get_leaves())
    sacking_index = sacking_index / (leaves**2) if leaves > 0 else 0
    return sacking_index

def colless_index_normalized(tree):
    def imbalance(node):
        if node.is_leaf():
            return 0
        children = node.get_children()
        if len(children) != 2:
            # allow multifurcations
            return sum(imbalance(c) for c in children)
        left = len(children[0].get_leaves())
        right = len(children[1].get_leaves())
        return abs(left - right) + imbalance(children[0]) + imbalance(children[1])

    C = imbalance(tree)
    n = len(tree.get_leaves())

    # maximum Colless index for binary tree with n leaves
    C_max = (n - 1) * (n - 2) / 2 if n >= 3 else 0

    return C / C_max if C_max > 0 else 0

#find. normalized version of colless index (divide by (n-1)2)

def compute_tree_metrics(datasets, model_name="Model"):
    results = []
    for ds in datasets:
        host_nwk = ds['host_newick']
        parasite_nwk = ds['parasite_newick']

        if not host_nwk.strip().endswith(';'):
            host_nwk += ';'
        if not parasite_nwk.strip().endswith(';'):
            parasite_nwk += ';'

        try:
            h_tree = Tree(host_nwk, format=1)
            p_tree = Tree(parasite_nwk, format=1)
        except Exception as e:
            print(f"[ERROR] Failed to parse trees in file: {ds['filename']} — {e}")
            continue

        results.append({
            'model': model_name,
            'filename': ds['filename'],
            'host_leaves': len(h_tree.get_leaves()),
            'parasite_leaves': len(p_tree.get_leaves()),
            'cherry_index_host': cherry_index(h_tree),
            'cherry_index_parasite': cherry_index(p_tree),
            'sackin_index_host': sackin_index(h_tree),
            'sackin_index_parasite': sackin_index(p_tree),
            'sackin_index_topology_host': sackin_index_topology(h_tree),
            'sackin_index_topology_parasite': sackin_index_topology(p_tree),
            'colless_index_host': colless_index_normalized(h_tree),
            'colless_index_parasite': colless_index_normalized(p_tree),
        })
    return pd.DataFrame(results)



def plot_metric_comparison_paired(df_all, base_metric_name, title=None, scenario=None):
    host_col = f"{base_metric_name}_host"
    parasite_col = f"{base_metric_name}_parasite"

    # Melt the data to long format
    df_melted = df_all.melt(
        id_vars=['model'],
        value_vars=[host_col, parasite_col],
        var_name='metric_type',
        value_name='value'
    )

    # Clean up metric_type for legend (host / parasite)
    df_melted['metric_type'] = df_melted['metric_type'].apply(
        lambda x: 'Host' if 'host' in x else 'Parasite'
    )

    # Plot
    plt.figure(figsize=(8, 6))
    sns.barplot(data=df_melted, x='model', y='value', hue='metric_type', errorbar='sd')

    plt.title(title or f'{base_metric_name.replace("_", " ").title()} (Host vs Parasite)')
    plt.ylabel(base_metric_name.replace('_', ' ').title())
    plt.xlabel('Model')
    plt.legend(title='Tree Type')
    plt.savefig(f'/Users/gabriele/synthetic_cophylo/comparison_trees/{scenario}/{base_metric_name}.png')
    plt.tight_layout()
    plt.show()


In [ ]:
# import tempfile
# import subprocess
# import os
# import re
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor, as_completed


# def run_parafit_on_dataset(dataset, r_script_path="parafit.R", nperm=500):
#     host_nwk = dataset['host_newick'].strip() + ";"
#     parasite_nwk = dataset['parasite_newick'].strip() + ";"
#     associations = dataset['associations']

#     with tempfile.TemporaryDirectory() as tmpdir:
#         host_path = os.path.join(tmpdir, "host.nwk")
#         parasite_path = os.path.join(tmpdir, "parasite.nwk")
#         assoc_path = os.path.join(tmpdir, "assoc.txt")

#         with open(host_path, "w") as f:
#             f.write(host_nwk)
#         with open(parasite_path, "w") as f:
#             f.write(parasite_nwk)
#         with open(assoc_path, "w") as f:
#             for p, h in associations:
#                 f.write(f"{p.strip().strip(',')}\t{h.strip().strip(',')}\n")

#         cmd = ["Rscript", r_script_path, host_path, parasite_path, assoc_path, str(nperm)]
#         try:
#             result = subprocess.check_output(cmd, text=True)
#             return (dataset['filename'], result)
#         except subprocess.CalledProcessError as e:
#             print(f"[ERROR] R script failed for {dataset['filename']}: {e.output}")
#             return (dataset['filename'], None)

# def extract_parafit_values(output_str):
#     stat_match = re.search(r"ParaFit Global Stat:\s*([\d\.eE+-]+)", output_str)
#     pval_match = re.search(r"ParaFit Global P-value:\s*([\d\.eE+-]+)", output_str)
#     if stat_match and pval_match:
#         return float(stat_match.group(1)), float(pval_match.group(1))
#     return None, None

# def run_all_parafit(datasets, model_name, max_workers=6):
#     results = []
#     print(f"Running ParaFit for {len(datasets)} datasets with {max_workers} threads")

#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = {
#             executor.submit(run_parafit_on_dataset, dataset): dataset
#             for dataset in datasets
#         }

#         for future in as_completed(futures):
#             dataset = futures[future]
#             filename = dataset['filename']
#             try:
#                 fname, output = future.result()
#                 if output:
#                     stat, pval = extract_parafit_values(output)
#                     if stat is not None:
#                         results.append({
#                             'model': model_name,
#                             'filename': fname,
#                             'parafit_stat': stat,
#                             'parafit_pval': pval
#                         })
#             except Exception as e:
#                 print(f"[ERROR] Exception in ParaFit for {filename}: {e}")

#     return pd.DataFrame(results)

In [ ]:
# extract reconcilation data
def extract_sim_stats(datasets):

    extracted = []

    for data in datasets:
        stats = data.get('sim_stats', {})
        fname = data.get('filename', 'unknown')

        # --- Case 1: new event-based format ---
        if any(k in stats for k in ["Cospeciations", "Host_Speciations", "Symbiont_Speciations"]):
            extracted.append({
                'filename': fname,
                'Total_Events': stats.get('Total_Events'),
                'Cospeciations': stats.get('Cospeciations'),
                'Host_Speciations': stats.get('Host_Speciations'),
                'Symbiont_Speciations': stats.get('Symbiont_Speciations'),
                'Host_Extinctions': stats.get('Host_Extinctions'),
                'Symbiont_Extinctions': stats.get('Symbiont_Extinctions'),
                'Host_Spreads_switches': stats.get('Host_Spreads_switches'),
            })

        elif any(k in stats for k in ["Expected", "Observed", "Frequencies"]):
            def parse_list(s):
                if isinstance(s, str):
                    s = s.strip("[]")
                    if not s:
                        return []
                    return [float(x) for x in s.split(",")]
                elif isinstance(s, list):
                    return s
                return []

            extracted.append({
                'filename': fname,
                'Probabilities': parse_list(stats.get('Probabilities', '')),
                'Frequencies': parse_list(stats.get('Frequencies', '')),
            })

        # --- Case 3: no recognizable stats ---
        else:
            extracted.append({'filename': fname, **stats})

    return extracted


def expand_event_columns(sim_stats_list):
    records = []
    event_names = ["Cospeciation", "Duplication", "Switch", "Loss"]

    for entry in sim_stats_list:
        fname = entry.get("filename")
        probs = entry.get("Probabilities", [None]*4)
        freqs = entry.get("Frequencies", [None]*4)

        # Handle cases with missing or shorter lists
        probs = list(probs) + [None]*(4 - len(probs))
        freqs = list(freqs) + [None]*(4 - len(freqs))

        record = {"filename": fname}
        for i, event in enumerate(event_names):
            record[f"Prob_{event}"] = probs[i]
            record[f"Freq_{event}"] = freqs[i]
        records.append(record)

    return pd.DataFrame(records)


def flatten_sim_stats(sim_stats_list):

    flattened = []

    for entry in sim_stats_list:
        record = {}
        record["filename"] = entry.get("filename", "unknown")

        for key, value in entry.items():
            if key == "filename":
                continue

            # If value is a list or tuple → expand it
            if isinstance(value, (list, tuple)):
                for i, v in enumerate(value):
                    record[f"{key}_{i}"] = v
            else:
                record[key] = value

        flattened.append(record)

    df = pd.DataFrame(flattened)
    cols = ["filename"] + [c for c in df.columns if c != "filename"]
    return df[cols]

# General functions for network analysis

In [ ]:
# functions to get bipartite graphs
import networkx as nx
import matplotlib.pyplot as plt

def build_bipartite_graph(dataset, show_plot=True, save_path=None):
    associations = dataset['associations']
    host_nodes = {h for _, h in associations}
    parasite_nodes = {p for p, _ in associations}

    G = nx.Graph()
    
    # Add nodes with bipartite attribute
    G.add_nodes_from(host_nodes, bipartite='host')
    G.add_nodes_from(parasite_nodes, bipartite='parasite')
    
    # Add edges
    G.add_edges_from(associations)

    if show_plot or save_path:
        plt.figure(figsize=(10, 6))
        pos = {}
        pos.update((n, (1, i)) for i, n in enumerate(sorted(host_nodes)))  # Hosts at y=1
        pos.update((n, (2, i)) for i, n in enumerate(sorted(parasite_nodes)))  # Parasites at y=2

        nx.draw(G, pos=pos, with_labels=True, node_color=["skyblue" if n in host_nodes else "salmon" for n in G.nodes()], edge_color="gray")
        plt.title(f"Bipartite Graph - {dataset['filename']}")
        plt.axis("off")
        
        if save_path:
            plt.savefig(save_path, bbox_inches="tight")
        if show_plot:
            plt.show()
        else:
            plt.close()

    return G

def generate_all_bipartite_graphs(datasets, show_plots=False, save_dir=None):

    import os

    graphs = {}
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    for dataset in datasets:
        filename = dataset['filename']
        save_path = f"{save_dir}/{filename.replace('.tgl', '')}_bipartite.png" if save_dir else None
        G = build_bipartite_graph(dataset, show_plot=show_plots, save_path=save_path)
        graphs[filename] = G

    return graphs

In [ ]:
# measure network metrics, such as density, average degree, assortativity, and centrality
import numpy as np
def get_node_sets(graph):
    hosts = [n for n, d in graph.nodes(data=True) if d.get("bipartite") == 'host']
    parasites = [n for n, d in graph.nodes(data=True) if d.get("bipartite") == 'parasite']
    return hosts, parasites


def analyze_bipartite_graph(graph, name="Unknown"):
    hosts, parasites = get_node_sets(graph)
    total_possible_edges = len(hosts) * len(parasites)

    degrees = dict(graph.degree())
    host_degrees = [degrees[n] for n in hosts]
    parasite_degrees = [degrees[n] for n in parasites]

    centrality = nx.degree_centrality(graph)
    top_central_nodes = sorted(centrality.items(), key=lambda x: -x[1])[:5]

    return {
        'dataset': name,
        'num_hosts': len(hosts),
        'num_parasites': len(parasites),
        'num_edges': graph.number_of_edges(),
        'density': graph.number_of_edges() / total_possible_edges if total_possible_edges else 0,
        'avg_host_degree': np.mean(host_degrees),
        'avg_parasite_degree': np.mean(parasite_degrees),
        'host_hotspots': sum(d > np.mean(host_degrees) + np.std(host_degrees) for d in host_degrees),
        'degree_assortativity': nx.degree_assortativity_coefficient(graph),
        'top_central_nodes': top_central_nodes,
        'num_components': nx.number_connected_components(graph),
    }

# Si può fare anche parasite hotspots
def analyze_model_graphs(graph_dict, model_name):
    return [analyze_bipartite_graph(g, name=f"{model_name}_{k}") for k, g in graph_dict.items()]

# Analyze Coala's trees

In [ ]:
import os

def parse_coala_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []
    sim_stats = {}


    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    stat_lines = []
    main_lines = []
    after_end_found = False
    for line in lines:
        if after_end_found:
            stat_lines.append(line)
        else:
            main_lines.append(line)
            if line.strip() == "END;":
                after_end_found = True

    for line in main_lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and "TREE" in stripped:
            host_newick = stripped.split("=")[1].strip(" ;")

        elif in_parasite_block and "TREE" in stripped:
            parasite_newick = stripped.split("=")[1].strip(" ;")

        # Extract RANGE mappings
        elif in_distribution_block and ":" in stripped:
            parasite, host = map(str.strip, stripped.split(":"))
            associations.append((parasite, host))

    # Parse statistics after END;
    for line in stat_lines:
        stripped = line.strip()
        if not stripped.startswith("#") or ":" not in stripped:
            continue
        try:
            key, val = map(str.strip, stripped[1:].split(":", 1))  # Remove leading '#' and split
            key = key.replace(" ", "_")
            val = val.strip()
            sim_stats[key] = val
        except Exception as e:
            print(f"Failed to parse stat line: {stripped} -> {e}")
            
    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations,
        "sim_stats": sim_stats
    }

    return data

def parse_all_coala_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tgl"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_coala_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets


## High Cospeciation

In [ ]:
datasets_coala_cosp = parse_all_coala_in_folder('//Users/gabriele/synthetic_cophylo/generate_coala/generated_trees/high_cosp/Datasets')
datasets_coala_cosp

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_coala_cosp:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
average_depth = average_host_tree_depth(datasets_coala_cosp)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_coala_cosp)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_coala_cosp)
print(f"Average host tree length (max depth): {average_length}")

average_length = average_parasite_tree_length(datasets_coala_cosp)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_coala_cosp)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_coala_cosp)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_coala_cosp)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_coala_cosp)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_coala_cosp)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_coala_cosp)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## High Switch

In [ ]:
datasets_coala_switch = parse_all_coala_in_folder('/Users/gabriele/synthetic_cophylo/generate_coala/generated_trees/high_switch/Datasets')
datasets_coala_switch

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_coala_switch:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
average_depth = average_host_tree_depth(datasets_coala_switch)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_coala_switch)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_coala_switch)
print(f"Average host tree length (max depth): {average_length}")    
average_length = average_parasite_tree_length(datasets_coala_switch)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_coala_switch)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_coala_switch)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_coala_switch)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_coala_switch)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_coala_switch)
print(f"Min number of associations in a dataset: {min_associations}")


multiple_assoc_counts = count_multiple_associations(datasets_coala_switch)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## Medium Values

In [ ]:
datasets_coala_med = parse_all_coala_in_folder('/Users/gabriele/synthetic_cophylo/generate_coala/generated_trees/medium/Datasets')
datasets_coala_med

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_coala_med:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
average_depth = average_host_tree_depth(datasets_coala_med)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_coala_med)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_coala_med)
print(f"Average host tree length (max depth): {average_length}")    
average_length = average_parasite_tree_length(datasets_coala_med)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_coala_med)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_coala_med)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_coala_med)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_coala_med)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_coala_med)
print(f"Min number of associations in a dataset: {min_associations}")


multiple_assoc_counts = count_multiple_associations(datasets_coala_switch)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## Comparison between scenarios


In [ ]:
import matplotlib.pyplot as plt
host_depths = [
    average_host_tree_depth(datasets_coala_cosp),
    average_host_tree_depth(datasets_coala_med),
    average_host_tree_depth(datasets_coala_switch)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_coala_cosp),
    average_parasite_tree_depth(datasets_coala_med),
    average_parasite_tree_depth(datasets_coala_switch)
]

# Plotting
labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Coala Model')
plt.legend()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/avg_branches.png', dpi=300)
plt.tight_layout()
plt.show()


In [ ]:


# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_coala_cosp),
    host_tree_branches_list(datasets_coala_med),
    host_tree_branches_list(datasets_coala_switch),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_coala_cosp),
    parasite_tree_branches_list(datasets_coala_med),
    parasite_tree_branches_list(datasets_coala_switch),
]

labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Coala Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/branches_distribution.png', dpi=300)


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()


In [ ]:

# Compute average lengths
host_lengths = [
    average_host_tree_leaves(datasets_coala_cosp),
    average_host_tree_leaves(datasets_coala_med),
    average_host_tree_leaves(datasets_coala_switch),    
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_leaves(datasets_coala_cosp),
    average_parasite_tree_leaves(datasets_coala_med),
    average_parasite_tree_leaves(datasets_coala_switch),  

]

# Plotting
labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Coala Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/avg_leaves.png', dpi=300)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

host_depths = [
    host_tree_leaves_list(datasets_coala_cosp),
    host_tree_leaves_list(datasets_coala_med),
    host_tree_leaves_list(datasets_coala_switch),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_coala_cosp),
    parasite_tree_leaves_list(datasets_coala_med),
    parasite_tree_leaves_list(datasets_coala_switch),
]

labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Coala Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/leaves_distribution.png', dpi=300)


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_coala_cosp),
    host_vs_parasite_list(datasets_coala_med),
    host_vs_parasite_list(datasets_coala_switch),
]

labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/dataset_branches.png', dpi=300)

plt.show()


In [ ]:


# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_coala_cosp),
    host_vs_parasite_list_leaves(datasets_coala_med),
    host_vs_parasite_list_leaves(datasets_coala_switch),
]

labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/dataset_leaves.png', dpi=300)

plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_coala_cosp),
    host_vs_parasite_list_length(datasets_coala_med),
    host_vs_parasite_list_length(datasets_coala_switch),
]

labels = ['Coala Cosp', 'Coala Medium', 'Coala Switch']


plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/dataset_length.png', dpi=300)

plt.show()


In [ ]:
import openpyxl
coala_cosp_metrics = compute_tree_metrics(datasets_coala_cosp, model_name="Coala_Cosp")
coala_medium_metrics = compute_tree_metrics(datasets_coala_med, model_name="Coala_Medium")
coala_switch_metrics = compute_tree_metrics(datasets_coala_switch, model_name="Coala_Switch")

comparison_metrics_cosp = pd.concat([coala_cosp_metrics, coala_medium_metrics, coala_switch_metrics])
comparison_metrics_cosp.to_excel('comparison_metrics_cosp.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index', scenario='coala')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index', scenario='coala')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology', scenario='coala')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index', scenario='coala')

In [ ]:
recon_cosp_coala = extract_sim_stats(datasets_coala_cosp)
recon_med_coala = extract_sim_stats(datasets_coala_med)
recon_switch_coala = extract_sim_stats(datasets_coala_switch)


flattened_cosp_coala = expand_event_columns(recon_cosp_coala).fillna(0)
flattened_med_coala = expand_event_columns(recon_med_coala).fillna(0)
flattened_switch_coala = expand_event_columns(recon_switch_coala).fillna(0)


boxplot_coala_cosp = flattened_cosp_coala[['filename', 'Freq_Cospeciation', 'Freq_Switch']]
boxplot_coala_cosp.rename(columns={'Freq_Cospeciation': 'Cospeciation', 'Freq_Switch': 'Host_switch'}, inplace=True)

boxplot_coala_med = flattened_med_coala[['filename', 'Freq_Cospeciation', 'Freq_Switch']]
boxplot_coala_med.rename(columns={'Freq_Cospeciation': 'Cospeciation', 'Freq_Switch': 'Host_switch'}, inplace=True)

boxplot_coala_switch = flattened_switch_coala[['filename', 'Freq_Cospeciation', 'Freq_Switch']]
boxplot_coala_switch.rename(columns={'Freq_Cospeciation': 'Cospeciation', 'Freq_Switch': 'Host_switch'}, inplace=True)


boxplot_coala_cosp["Model"] = "Coala Cospeciation"
boxplot_coala_med["Model"] = "Coala Median"
boxplot_coala_switch["Model"] = "Coala Switch"


# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_coala_cosp, boxplot_coala_med, boxplot_coala_switch],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Scenarios [Coala]", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/coala/reconciliation.png', dpi=300)

plt.tight_layout()
plt.show()

# Analyze Treeducken's trees

In [ ]:
import os 
def parse_treeducken_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []
    sim_stats = {}

    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    for line in lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and "TREE" in stripped:
            host_newick = stripped.split("=")[1].strip(" ;")

        elif in_parasite_block and "TREE" in stripped:
            parasite_newick = stripped.split("=")[1].strip(" ;")

        # Extract RANGE mappings
        elif in_distribution_block and ":" in stripped:
            parasite, host = map(str.strip, stripped.split(":"))
            associations.append((parasite, host))

    # Parse statistics from end lines (formatted as: Key [space] Value)
    stat_pattern = re.compile(r"^([A-Za-z_/]+[\w\s/]*?)\s+([-\d.eE]+|NaN)$")
    for line in lines[::-1]:  # reverse iterate to focus on end of file
        match = stat_pattern.match(line.strip())
        if match:
            key = match.group(1).strip().replace(" ", "_").replace("/", "_")
            val_str = match.group(2).strip()
            try:
                val = float(val_str) if val_str.lower() != "nan" else None
                sim_stats[key] = val
            except ValueError:
                print(f"Could not convert value to float for line: {line.strip()}")

    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations,
        "sim_stats": sim_stats
    }

    return data

def parse_all_treeducken_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tgl"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_treeducken_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets

## High Cospeciation

In [ ]:
datasets_treeducken_cosp = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highCosp/Datasets')
datasets_treeducken_cosp

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_treeducken_cosp:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
average_depth = average_host_tree_depth(datasets_treeducken_cosp)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_treeducken_cosp)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_treeducken_cosp)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_treeducken_cosp)
print(f"Average parasite tree length (max depth): {average_length}")


average_leaves = average_host_tree_leaves(datasets_treeducken_cosp)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_treeducken_cosp)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_treeducken_cosp)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_treeducken_cosp)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_treeducken_cosp)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_treeducken_cosp)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## High Switch

In [ ]:
datasets_treeducken_switch = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highSwitch/Datasets')
datasets_treeducken_switch

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_treeducken_switch:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
average_depth = average_host_tree_depth(datasets_treeducken_switch)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_treeducken_switch)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_treeducken_switch)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_treeducken_switch)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_treeducken_switch)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_treeducken_switch)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_treeducken_switch)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_treeducken_switch)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_treeducken_switch)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_treeducken_switch)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")


## Medium Values

In [ ]:
datasets_treeducken_medium = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_medium/Datasets')
datasets_treeducken_medium

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_treeducken_medium:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree_stats']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    

average_depth = average_host_tree_depth(datasets_treeducken_medium)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_treeducken_medium)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_treeducken_medium)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_treeducken_medium)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_treeducken_medium)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_treeducken_medium)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations


In [ ]:
avg_association = average_associations_per_dataset(datasets_treeducken_medium)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_treeducken_medium)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_treeducken_medium)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_treeducken_medium)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")



## Comparison between scenarios

In [ ]:
#print distribution of multiple associations across datasets boxplot for medium, high swithc and high cosp
import matplotlib.pyplot as plt
import seaborn as sns
def plot_multiple_association_distribution(datasets_list, labels, model_name='Treeducken'):
    data = []
    for datasets, label in zip(datasets_list, labels):
        counts = count_multiple_associations(datasets)
        for count in counts:
            data.append({'Dataset': label, 'Multiple_Associations': count}) 
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Dataset', y='Multiple_Associations', data=df)
    plt.title(f'Distribution of Multiple Associations Across Datasets [{model_name}]')
    plt.ylabel('Number of Parasites with Multiple Associations')
    plt.savefig(f'/Users/gabriele/synthetic_cophylo/comparison_trees/{model_name}/multiple_associations.png', dpi=300)

    plt.xlabel('Dataset Type')
    plt.show()  
    
plot_multiple_association_distribution(
    [datasets_treeducken_cosp, datasets_treeducken_medium, datasets_treeducken_switch],
    ['High Cosp', 'Medium', 'High Switch']
)

In [ ]:
host_depths = [
    average_host_tree_depth(datasets_treeducken_cosp),
    average_host_tree_depth(datasets_treeducken_medium),
    average_host_tree_depth(datasets_treeducken_switch)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_treeducken_cosp),
    average_parasite_tree_depth(datasets_treeducken_medium),
    average_parasite_tree_depth(datasets_treeducken_switch)
]

# Plotting
labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Treeducken Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/avg_branches.png', dpi=300)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_treeducken_cosp),
    host_tree_branches_list(datasets_treeducken_medium),
    host_tree_branches_list(datasets_treeducken_switch),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_treeducken_cosp),
    parasite_tree_branches_list(datasets_treeducken_medium),
    parasite_tree_branches_list(datasets_treeducken_switch),
]

labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Treeducken Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/branches_distribution.png', dpi=300)


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute average lengths
host_lengths = [
    average_host_tree_leaves(datasets_treeducken_cosp),
    average_host_tree_leaves(datasets_treeducken_medium),
    average_host_tree_leaves(datasets_treeducken_switch),    
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_leaves(datasets_treeducken_cosp),
    average_parasite_tree_leaves(datasets_treeducken_medium),
    average_parasite_tree_leaves(datasets_treeducken_switch),  

]

# Plotting
labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Treeducken Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/avg_leaves.png', dpi=300)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

host_depths = [
    host_tree_leaves_list(datasets_treeducken_cosp),
    host_tree_leaves_list(datasets_treeducken_medium),
    host_tree_leaves_list(datasets_treeducken_switch),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_treeducken_cosp),
    parasite_tree_leaves_list(datasets_treeducken_medium),
    parasite_tree_leaves_list(datasets_treeducken_switch),
]

labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Treeducken Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/leaves_distribution.png', dpi=300)


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_treeducken_cosp),
    host_vs_parasite_list(datasets_treeducken_medium),
    host_vs_parasite_list(datasets_treeducken_switch),
]

labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/dataset_branches.png', dpi=300)

plt.show()



In [ ]:

# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_treeducken_cosp),
    host_vs_parasite_list_leaves(datasets_treeducken_medium),
    host_vs_parasite_list_leaves(datasets_treeducken_switch),
]

labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/dataset_leaves.png', dpi=300)

plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_treeducken_cosp),
    host_vs_parasite_list_length(datasets_treeducken_medium),
    host_vs_parasite_list_length(datasets_treeducken_switch),
]

labels = ['Treeducken Cosp', 'Treeducken Medium', 'Treeducken Switch']


plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/dataset_length.png', dpi=300)

plt.show()

In [ ]:
import openpyxl
treeducken_cosp_metrics = compute_tree_metrics(datasets_treeducken_cosp, model_name="Treeducken_Cosp")
treeducken_medium_metrics = compute_tree_metrics(datasets_treeducken_medium, model_name="Treeducken_Medium")
treeducken_switch_metrics = compute_tree_metrics(datasets_treeducken_switch, model_name="Treeducken_Switch")

comparison_metrics_cosp = pd.concat([treeducken_cosp_metrics, treeducken_medium_metrics, treeducken_switch_metrics])
comparison_metrics_cosp.to_excel('comparison_metrics_cosp.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index', scenario='treeducken')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index', scenario='treeducken')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology', scenario='treeducken')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index', scenario='treeducken')

In [ ]:
recon_cosp_treeducken = extract_sim_stats(datasets_treeducken_cosp)
recon_medium_treeducken = extract_sim_stats(datasets_treeducken_medium)
recon_switch_treeducken = extract_sim_stats(datasets_treeducken_switch)

flattened_cosp_treeducken = flatten_sim_stats(recon_cosp_treeducken).fillna(0)
flattened_medium_treeducken = flatten_sim_stats(recon_medium_treeducken).fillna(0)
flattened_switch_treeducken = flatten_sim_stats(recon_switch_treeducken).fillna(0)

boxplot_treeducken_cosp = flattened_cosp_treeducken[['filename', 'Cospeciations', 'Host_Spreads_switches']]
boxplot_treeducken_cosp.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spreads_switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_medium = flattened_medium_treeducken[['filename', 'Cospeciations', 'Host_Spreads_switches']]
boxplot_treeducken_medium.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spreads_switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_switch = flattened_switch_treeducken[['filename', 'Cospeciations', 'Host_Spreads_switches']]
boxplot_treeducken_switch.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spreads_switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_cosp["Model"] = "Treeducken Cospeciation"
boxplot_treeducken_medium["Model"] = "Treeducken Medium"
boxplot_treeducken_switch["Model"] = "Treeducken Switch"

# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_treeducken_cosp, boxplot_treeducken_medium, boxplot_treeducken_switch],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Scenarios [Treeducken]", fontsize=14, pad=15)
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/treeducken/reconciliation.png', dpi=300)

plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Analyze Cophylo's trees

In [ ]:
def parse_alcala_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []
    sim_stats = {}

    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    for line in lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST") or stripped.startswith("# HOST_TREE"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE") or stripped.startswith("# PARASITE_TREE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION") or stripped.startswith("# ASSOCIATIONS"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and stripped and not stripped.startswith("#"):
            host_newick = stripped.strip(" ;")

        elif in_parasite_block and stripped and not stripped.startswith("#"):
            parasite_newick = stripped.strip(" ;")

        # Extract associations
        elif in_distribution_block and not stripped.startswith("#") and " " in stripped:
            parts = stripped.split()
            if len(parts) == 2:
                parasite, host = parts
                associations.append((parasite, host))

    # Parse PARAMETERS block
    param_stats = {}
    in_param_block = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("# PARAMETERS"):
            in_param_block = True
            continue
        if stripped.startswith("ENDBLOCK") and in_param_block:
            in_param_block = False
            continue

        if in_param_block and "=" in stripped:
            key, value = map(str.strip, stripped.split("=", 1))
            key = key.replace(" ", "_")
            try:
                val = float(value)
                param_stats[key] = val
            except ValueError:
                param_stats[key] = value

    sim_stats = param_stats

    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations,
        "sim_stats": sim_stats
    }

    return data

def parse_all_alcala_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tgl"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_alcala_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets


## High Cospeciation


In [ ]:
datasets_alcala_cosp = parse_all_alcala_in_folder('/Users/gabriele/synthetic_cophylo/generate_alcala/generated_trees_highCosp/Dataset')
datasets_alcala_cosp

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_alcala_cosp:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
average_depth = average_host_tree_depth(datasets_alcala_cosp)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_alcala_cosp)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_alcala_cosp)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_alcala_cosp)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_alcala_cosp)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_alcala_cosp)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_alcala_cosp)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_alcala_cosp)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_alcala_cosp)
print(f"Min number of associations in a dataset: {min_associations}")

## High Switch

In [ ]:
datasets_alcala_switch = parse_all_alcala_in_folder('/Users/gabriele/synthetic_cophylo/generate_alcala/generated_trees_highSwitch/Dataset')
datasets_alcala_switch

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_alcala_switch:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    
average_depth = average_host_tree_depth(datasets_alcala_switch)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_alcala_switch)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_alcala_switch)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_alcala_switch)
print(f"Average parasite tree length (max depth): {average_length}")


average_leaves = average_host_tree_leaves(datasets_alcala_switch)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_alcala_switch)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_alcala_switch)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_alcala_switch)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_alcala_switch)
print(f"Min number of associations in a dataset: {min_associations}")

## Medium Values



In [ ]:
datasets_alcala_medium = parse_all_alcala_in_folder('/Users/gabriele/synthetic_cophylo/generate_alcala/generated_trees_medium/Dataset')
datasets_alcala_medium

### Analyze Host and Parasite trees

In [ ]:
for dataset in datasets_alcala_medium:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    
average_depth = average_host_tree_depth(datasets_alcala_medium)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_alcala_medium)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_alcala_medium)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_alcala_medium)
print(f"Average parasite tree length (max depth): {average_length}")


average_leaves = average_host_tree_leaves(datasets_alcala_medium)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_alcala_medium)
print(f"Average number of leaves in parasite trees: {average_leaves}")

### Analyze Associations

In [ ]:
avg_association = average_associations_per_dataset(datasets_alcala_medium)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_alcala_medium)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_alcala_medium)
print(f"Min number of associations in a dataset: {min_associations}")

## Comparison between scenarios

In [ ]:
import matplotlib.pyplot as plt

host_depths = [
    average_host_tree_depth(datasets_alcala_cosp),
    average_host_tree_depth(datasets_alcala_medium),
    average_host_tree_depth(datasets_alcala_switch)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_alcala_cosp),
    average_parasite_tree_depth(datasets_alcala_medium),
    average_parasite_tree_depth(datasets_alcala_switch)
]

# Plotting
labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Cophylo Model')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/avg_branches.png')
plt.show()


In [ ]:

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_alcala_cosp),
    host_tree_branches_list(datasets_alcala_medium),
    host_tree_branches_list(datasets_alcala_switch),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_alcala_cosp),
    parasite_tree_branches_list(datasets_alcala_medium),
    parasite_tree_branches_list(datasets_alcala_switch),
]

labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Cophylo Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/branches_distribution.png')


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Compute average lengths
host_lengths = [
    average_host_tree_leaves(datasets_alcala_cosp),
    average_host_tree_leaves(datasets_alcala_medium),
    average_host_tree_leaves(datasets_alcala_switch),    
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_leaves(datasets_alcala_cosp),
    average_parasite_tree_leaves(datasets_alcala_medium),
    average_parasite_tree_leaves(datasets_alcala_switch),  

]

# Plotting
labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Cophylo Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/average_leaves.png')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt

host_depths = [
    host_tree_leaves_list(datasets_alcala_cosp),
    host_tree_leaves_list(datasets_alcala_medium),
    host_tree_leaves_list(datasets_alcala_switch),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_alcala_cosp),
    parasite_tree_leaves_list(datasets_alcala_medium),
    parasite_tree_leaves_list(datasets_alcala_switch),
]

labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Cophylo Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/leaves_distribution.png')

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:


# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_alcala_cosp),
    host_vs_parasite_list(datasets_alcala_medium),
    host_vs_parasite_list(datasets_alcala_switch),
]

labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/dataset_branches.png')

plt.show()


In [ ]:



# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_alcala_cosp),
    host_vs_parasite_list_leaves(datasets_alcala_medium),
    host_vs_parasite_list_leaves(datasets_alcala_switch),
]

labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/dataset_leaves.png')

plt.show()


In [ ]:

# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_alcala_cosp),
    host_vs_parasite_list_length(datasets_alcala_medium),
    host_vs_parasite_list_length(datasets_alcala_switch),
]

labels = ['Cophylo Cosp', 'Cophylo Medium', 'Cophylo Switch']


plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/dataset_length.png')

plt.show()


### Analyze Associations


In [ ]:
import openpyxl
alcala_cosp_metrics = compute_tree_metrics(datasets_alcala_cosp, model_name="Cophylo_Cosp")
alcala_medium_metrics = compute_tree_metrics(datasets_alcala_medium, model_name="Cophylo_Medium")
alcala_switch_metrics = compute_tree_metrics(datasets_alcala_switch, model_name="Cophylo_Switch")

comparison_metrics_cosp = pd.concat([alcala_cosp_metrics, alcala_medium_metrics, alcala_switch_metrics])
comparison_metrics_cosp.to_excel('comparison_metrics_cosp.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index', scenario='cophylo')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index', scenario='cophylo')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology', scenario='cophylo')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index', scenario='cophylo')

In [ ]:
recon_cosp_alcala = extract_sim_stats(datasets_alcala_cosp)
recon_med_alcala = extract_sim_stats(datasets_alcala_medium)
recon_switch_alcala = extract_sim_stats(datasets_alcala_switch)


flattened_cosp_alcala = flatten_sim_stats(recon_cosp_alcala).fillna(0)
flattened_med_alcala = flatten_sim_stats(recon_med_alcala).fillna(0)
flattened_switch_alcala = flatten_sim_stats(recon_switch_alcala).fillna(0)


boxplot_alcala_cosp = flattened_cosp_alcala[['filename', 'Cospeciation', 'Host_switch']]
boxplot_alcala_med = flattened_med_alcala[['filename', 'Cospeciation', 'Host_switch']]
boxplot_alcala_switch = flattened_switch_alcala[['filename', 'Cospeciation', 'Host_switch']]


boxplot_alcala_cosp["Model"] = "Cophylo Cospeciation"
boxplot_alcala_med["Model"] = "Cophylo Medium"
boxplot_alcala_switch["Model"] = "Cophylo Switch"

# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_alcala_cosp, boxplot_alcala_med, boxplot_alcala_switch],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Scenarios [Cophylo]", fontsize=14, pad=15)
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/cophylo/reconciliation.png')
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Analyze Asymmetree's trees


In [ ]:
def parse_asymmetree_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []
    sim_stats = {}

    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    for line in lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK") | stripped.startswith("END;"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and "TREE" in stripped:
            host_newick = stripped.split("=")[1].strip(" ;")

        elif in_parasite_block and "TREE" in stripped:
            parasite_newick = stripped.split("=")[1].strip(" ;")

        # Extract RANGE mappings
        elif in_distribution_block and ":" in stripped:
            parasite, host = map(str.strip, stripped.split(":"))
            associations.append((parasite, host))

        # Parse statistics at bottom of .tgl file (e.g. "Loss: 5")
        stat_pattern = re.compile(r"^([A-Za-z\s]+):\s*([-\d.eE]+|NaN)$")

        for line in lines[::-1]:  # reverse to search bottom first
            line = line.strip()
            match = stat_pattern.match(line)
            if match:
                key = match.group(1).strip().replace(" ", "_")
                val_str = match.group(2).strip()
                try:
                    val = float(val_str) if val_str.lower() != "nan" else None
                    sim_stats[key] = val
                except ValueError:
                    print(f"[Warning] Could not parse numeric value in: {line}")

    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations,
        "sim_stats": sim_stats
    }

    return data

def parse_all_asymmetree_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tgl"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_asymmetree_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets


## Low Switch 


In [ ]:
datasets_asymmetree_cosp = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_low_switch/Datasets')
datasets_asymmetree_cosp


### Analyze Host and Parasite trees


In [ ]:
for dataset in datasets_asymmetree_cosp:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    
average_depth = average_host_tree_depth(datasets_asymmetree_cosp)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_asymmetree_cosp)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_asymmetree_cosp)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_asymmetree_cosp)
print(f"Average parasite tree length (max depth): {average_length}")


average_leaves = average_host_tree_leaves(datasets_asymmetree_cosp)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_asymmetree_cosp)
print(f"Average number of leaves in parasite trees: {average_leaves}")


### Analyze Associations


In [ ]:
avg_association = average_associations_per_dataset(datasets_asymmetree_cosp)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_asymmetree_cosp)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_asymmetree_cosp)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_asymmetree_cosp)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")


## High Switch


In [ ]:
datasets_asymmetree_switch = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_high_switch/Datasets')
datasets_asymmetree_switch

### Analyze Host and Parasite trees


In [ ]:
for dataset in datasets_asymmetree_switch:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    
average_depth = average_host_tree_depth(datasets_asymmetree_switch)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_asymmetree_switch)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_asymmetree_switch)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_asymmetree_switch)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_asymmetree_switch)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_asymmetree_switch)
print(f"Average number of leaves in parasite trees: {average_leaves}")


### Analyze Associations


In [ ]:
avg_association = average_associations_per_dataset(datasets_asymmetree_switch)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_asymmetree_switch)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_asymmetree_switch)
print(f"Min number of associations in a dataset: {min_associations}")


multiple_assoc_counts = count_multiple_associations(datasets_asymmetree_switch)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## Medium Values


In [ ]:
datasets_asymmetree_medium = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_medium/Datasets')
datasets_asymmetree_medium


### Analyze Host and Parasite trees


In [ ]:
for dataset in datasets_asymmetree_medium:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])

average_depth = average_host_tree_depth(datasets_asymmetree_medium)
print(f"Average host tree depth (# branches): {average_depth}")

average_depth = average_parasite_tree_depth(datasets_asymmetree_medium)
print(f"Average parasite tree depth (# branches): {average_depth}")

average_length = average_host_tree_length(datasets_asymmetree_medium)
print(f"Average host tree length (max depth): {average_length}")
average_length = average_parasite_tree_length(datasets_asymmetree_medium)
print(f"Average parasite tree length (max depth): {average_length}")

average_leaves = average_host_tree_leaves(datasets_asymmetree_medium)
print(f"Average number of leaves in host trees: {average_leaves}")

average_leaves = average_parasite_tree_leaves(datasets_asymmetree_medium)
print(f"Average number of leaves in parasite trees: {average_leaves}")


### Analyze Associations


In [ ]:
avg_association = average_associations_per_dataset(datasets_asymmetree_medium)
print(f"Average number of associations per dataset: {avg_association}")

# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in datasets_asymmetree_medium)
print(f"Max number of associations in a dataset: {max_associations}")

# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in datasets_asymmetree_medium)
print(f"Min number of associations in a dataset: {min_associations}")

multiple_assoc_counts = count_multiple_associations(datasets_asymmetree_medium)
average_multiple_assoc = sum(multiple_assoc_counts) / len(multiple_assoc_counts)
print(f"Average number of parasites with multiple associations per dataset: {average_multiple_assoc}")

## Comparison between scenarios

In [ ]:
plot_multiple_association_distribution(
    [datasets_asymmetree_cosp, datasets_asymmetree_medium, datasets_asymmetree_switch],
    ['High Cosp', 'Medium', 'High Switch'], model_name='asymmetree')

In [ ]:
host_depths = [
    average_host_tree_depth(datasets_asymmetree_cosp),
    average_host_tree_depth(datasets_asymmetree_medium),
    average_host_tree_depth(datasets_asymmetree_switch)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_asymmetree_cosp),
    average_parasite_tree_depth(datasets_asymmetree_medium),
    average_parasite_tree_depth(datasets_asymmetree_switch)
]

# Plotting
labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per AsymmeTree Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/avg_branches.png')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_asymmetree_cosp),
    host_tree_branches_list(datasets_asymmetree_medium),
    host_tree_branches_list(datasets_asymmetree_switch),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_asymmetree_cosp),
    parasite_tree_branches_list(datasets_asymmetree_medium),
    parasite_tree_branches_list(datasets_asymmetree_switch),
]

labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across AsymmeTree Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/branches_distribution.png')


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute average lengths
host_lengths = [
    average_host_tree_leaves(datasets_asymmetree_cosp),
    average_host_tree_leaves(datasets_asymmetree_medium),
    average_host_tree_leaves(datasets_asymmetree_switch),    
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_leaves(datasets_asymmetree_cosp),
    average_parasite_tree_leaves(datasets_asymmetree_medium),
    average_parasite_tree_leaves(datasets_asymmetree_switch),  

]

# Plotting
labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per AsymmeTree Model')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/avg_leaves.png')

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

host_depths = [
    host_tree_leaves_list(datasets_asymmetree_cosp),
    host_tree_leaves_list(datasets_asymmetree_medium),
    host_tree_leaves_list(datasets_asymmetree_switch),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_asymmetree_cosp),
    parasite_tree_leaves_list(datasets_asymmetree_medium),
    parasite_tree_leaves_list(datasets_asymmetree_switch),
]

labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']
plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Asymmetree Models")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/leaves_distribution.png')


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_asymmetree_cosp),
    host_vs_parasite_list(datasets_asymmetree_medium),
    host_vs_parasite_list(datasets_asymmetree_switch),
]

labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/dataset_branches.png')

plt.show()


# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_asymmetree_cosp),
    host_vs_parasite_list_leaves(datasets_asymmetree_medium),
    host_vs_parasite_list_leaves(datasets_asymmetree_switch),
]

labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/dataset_leaves.png')
plt.show()

# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_asymmetree_cosp),
    host_vs_parasite_list_length(datasets_asymmetree_medium),
    host_vs_parasite_list_length(datasets_asymmetree_switch),
]

labels = ['AsymmeTree Cosp', 'AsymmeTree Medium', 'AsymmeTree Switch']


plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/dataset_length.png')
plt.show()

In [ ]:
import openpyxl
asymmetree_cosp_metrics = compute_tree_metrics(datasets_asymmetree_cosp, model_name="AsymmeTree_Cosp")
asymmetree_medium_metrics = compute_tree_metrics(datasets_asymmetree_medium, model_name="AsymmeTree_Medium")
asymmetree_switch_metrics = compute_tree_metrics(datasets_asymmetree_switch, model_name="AsymmeTree_Switch")

comparison_metrics_cosp = pd.concat([asymmetree_cosp_metrics, asymmetree_medium_metrics, asymmetree_switch_metrics])
comparison_metrics_cosp.to_excel('comparison_metrics_cosp.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index', scenario='asymmetree')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index', scenario='asymmetree')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology', scenario='asymmetree')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index', scenario='asymmetree')

In [ ]:
recon_cosp_asymmetree = extract_sim_stats(datasets_asymmetree_cosp)

flattened_cosp_asymmetree = flatten_sim_stats(recon_cosp_asymmetree).fillna(0)

flattened_cosp_asymmetree['Total Events'] = (flattened_cosp_asymmetree['Horizontal_Gene_Transfer'] +
    flattened_cosp_asymmetree['Duplication'] +
    flattened_cosp_asymmetree['Speciation'] +
    flattened_cosp_asymmetree['Loss'])
flattened_cosp_asymmetree['Horizontal_Gene_Transfer_Rate'] = flattened_cosp_asymmetree['Horizontal_Gene_Transfer'] / flattened_cosp_asymmetree['Total Events']
flattened_cosp_asymmetree['Duplication_Rate'] = flattened_cosp_asymmetree['Duplication'] / flattened_cosp_asymmetree['Total Events']
flattened_cosp_asymmetree['Speciation_Rate'] = flattened_cosp_asymmetree['Speciation'] / flattened_cosp_asymmetree['Total Events']
flattened_cosp_asymmetree['Loss_Rate'] = flattened_cosp_asymmetree['Loss'] / flattened_cosp_asymmetree['Total Events']
flattened_cosp_asymmetree.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_cosp = flattened_cosp_asymmetree[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_cosp.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)


boxplot_asymmetree_cosp["Model"] = "AsymmeTree Cospeciation"

recon_medium_asymmetree = extract_sim_stats(datasets_asymmetree_medium)

flattened_medium_asymmetree = flatten_sim_stats(recon_medium_asymmetree).fillna(0)

flattened_medium_asymmetree['Total Events'] = (flattened_medium_asymmetree['Horizontal_Gene_Transfer'] +
    flattened_medium_asymmetree['Duplication'] +
    flattened_medium_asymmetree['Speciation'] +
    flattened_medium_asymmetree['Loss'])
flattened_medium_asymmetree['Horizontal_Gene_Transfer_Rate'] = flattened_medium_asymmetree['Horizontal_Gene_Transfer'] / flattened_medium_asymmetree['Total Events']
flattened_medium_asymmetree['Duplication_Rate'] = flattened_medium_asymmetree['Duplication'] / flattened_medium_asymmetree['Total Events']
flattened_medium_asymmetree['Speciation_Rate'] = flattened_medium_asymmetree['Speciation'] / flattened_medium_asymmetree['Total Events']
flattened_medium_asymmetree['Loss_Rate'] = flattened_medium_asymmetree['Loss'] / flattened_medium_asymmetree['Total Events']
flattened_medium_asymmetree.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_medium = flattened_medium_asymmetree[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_medium.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)


boxplot_asymmetree_medium["Model"] = "AsymmeTree Medium"



recon_switch_asymmetree = extract_sim_stats(datasets_asymmetree_switch)

flattened_switch_asymmetree = flatten_sim_stats(recon_switch_asymmetree).fillna(0)

flattened_switch_asymmetree['Total Events'] = (flattened_switch_asymmetree['Horizontal_Gene_Transfer'] +
    flattened_switch_asymmetree['Duplication'] +
    flattened_switch_asymmetree['Speciation'] +
    flattened_switch_asymmetree['Loss'])
flattened_switch_asymmetree['Horizontal_Gene_Transfer_Rate'] = flattened_switch_asymmetree['Horizontal_Gene_Transfer'] / flattened_switch_asymmetree['Total Events']
flattened_switch_asymmetree['Duplication_Rate'] = flattened_switch_asymmetree['Duplication'] / flattened_switch_asymmetree['Total Events']
flattened_switch_asymmetree['Speciation_Rate'] = flattened_switch_asymmetree['Speciation'] / flattened_switch_asymmetree['Total Events']
flattened_switch_asymmetree['Loss_Rate'] = flattened_switch_asymmetree['Loss'] / flattened_switch_asymmetree['Total Events']
flattened_switch_asymmetree.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_switch = flattened_switch_asymmetree[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_switch.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)


boxplot_asymmetree_switch["Model"] = "AsymmeTree Switch"




# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_asymmetree_cosp, boxplot_asymmetree_medium, boxplot_asymmetree_switch],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across scenarios [AsymmeTree]", fontsize=14, pad=15)
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/asymmetree/reconciliation.png')

plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Analyze Real Data's trees


Find hotspots for real data in all datasets 

In [ ]:
def parse_realdata_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []

    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    stat_lines = []
    main_lines = []
    after_end_found = False
    for line in lines:
        if after_end_found:
            stat_lines.append(line)
        else:
            main_lines.append(line)
            if line.strip() == "END;":
                after_end_found = True

    for line in main_lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and "TREE" in stripped:
            host_newick = stripped.split("=")[1].strip(" ;")

        elif in_parasite_block and "TREE" in stripped:
            parasite_newick = stripped.split("=")[1].strip(" ;")

        # Extract RANGE mappings
        elif in_distribution_block and ":" in stripped:
            parasite, host = map(str.strip, stripped.split(":"))
            associations.append((parasite, host))


    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations
    }

    return data


def parse_all_realdata_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".nex"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_realdata_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets


In [ ]:
real_dataset = parse_all_realdata_in_folder('/Users/gabriele/synthetic_cophylo/real_data')
real_dataset

### Analyze Host and Parasite trees


In [ ]:
for dataset in real_dataset:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
average_depth = average_host_tree_depth(real_dataset)
print(f"Average host tree depth (# branches): {average_depth}")

In [ ]:
average_depth = average_parasite_tree_depth(real_dataset)
print(f"Average parasite tree depth (# branches): {average_depth}")


In [ ]:

average_leaves = average_host_tree_leaves(real_dataset)
print(f"Average number of leaves in host trees: {average_leaves}")


In [ ]:

average_leaves = average_parasite_tree_leaves(real_dataset)
print(f"Average number of leaves in parasite trees: {average_leaves}")


### Analyze Associations


In [ ]:
avg_association = average_associations_per_dataset(real_dataset)
print(f"Average number of associations per dataset: {avg_association}")

In [ ]:
# print max number of associations in a dataset
max_associations = max(len(dataset['associations']) for dataset in real_dataset)
print(f"Max number of associations in a dataset: {max_associations}")

In [ ]:
# print min number of associations in a dataset
min_associations = min(len(dataset['associations']) for dataset in real_dataset)
print(f"Min number of associations in a dataset: {min_associations}")

# Comparsion between models

## Compare trees

### High Cospeciation

- Compare depths of trees using number of branches

In [ ]:
host_depths = [
    average_host_tree_depth(datasets_coala_cosp),
    average_host_tree_depth(datasets_treeducken_cosp),
    average_host_tree_depth(datasets_alcala_cosp),
    average_host_tree_depth(datasets_asymmetree_cosp),
    #average_host_tree_depth(real_dataset)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_coala_cosp),
    average_parasite_tree_depth(datasets_treeducken_cosp),
    average_parasite_tree_depth(datasets_alcala_cosp),
    average_parasite_tree_depth(datasets_asymmetree_cosp),
    #average_parasite_tree_depth(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Model with High Cospeciation')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/avg_branches.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_coala_cosp),
    host_tree_branches_list(datasets_treeducken_cosp),
    host_tree_branches_list(datasets_alcala_cosp),
    host_tree_branches_list(datasets_asymmetree_cosp),
    #host_tree_branches_list(real_dataset),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_coala_cosp),
    parasite_tree_branches_list(datasets_treeducken_cosp),
    parasite_tree_branches_list(datasets_alcala_cosp),
    parasite_tree_branches_list(datasets_asymmetree_cosp),
    #parasite_tree_branches_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/branches_distribution.png', dpi=300)
plt.show()

- Compare trees using depth and branches lengths

In [ ]:
import matplotlib.pyplot as plt

# Compute average lengths
host_lengths = [
    average_host_tree_length(datasets_coala_cosp),
    average_host_tree_length(datasets_treeducken_cosp),
    average_host_tree_length(datasets_alcala_cosp),
    average_host_tree_length(datasets_asymmetree_cosp),
    
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_length(datasets_coala_cosp),
    average_parasite_tree_length(datasets_treeducken_cosp),
    average_parasite_tree_length(datasets_alcala_cosp),
    average_parasite_tree_length(datasets_asymmetree_cosp)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Length')
plt.title('Average Length (max depth) of Host and Parasite Trees per Model with High Cosp')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/avg_length.png', dpi=300)
plt.show()

- Compare trees using number of leaves

In [ ]:
host_leaves = [
    average_host_tree_leaves(datasets_coala_cosp),
    average_host_tree_leaves(datasets_treeducken_cosp),
    average_host_tree_leaves(datasets_alcala_cosp),
    average_host_tree_leaves(datasets_asymmetree_cosp),
    #average_host_tree_leaves(real_dataset)
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_coala_cosp),
    average_parasite_tree_leaves(datasets_treeducken_cosp),
    average_parasite_tree_leaves(datasets_alcala_cosp),
    average_parasite_tree_leaves(datasets_asymmetree_cosp),
    #average_parasite_tree_leaves(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Model with High Cospeciation')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/avg_leaves.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_leaves_list(datasets_coala_cosp),
    host_tree_leaves_list(datasets_treeducken_cosp),
    host_tree_leaves_list(datasets_alcala_cosp),
    host_tree_leaves_list(datasets_asymmetree_cosp),
    #host_tree_leaves_list(real_dataset),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_coala_cosp),
    parasite_tree_leaves_list(datasets_treeducken_cosp),
    parasite_tree_leaves_list(datasets_alcala_cosp),
    parasite_tree_leaves_list(datasets_asymmetree_cosp),
    #parasite_tree_leaves_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/leaves_distribution.png', dpi=300)
plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_coala_cosp),
    host_vs_parasite_list(datasets_treeducken_cosp),
    host_vs_parasite_list(datasets_alcala_cosp),
    host_vs_parasite_list(datasets_asymmetree_cosp),
    #host_vs_parasite_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences [High Cospeciation]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/dataset_branches.png', dpi=300)
plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_coala_cosp),
    host_vs_parasite_list_leaves(datasets_treeducken_cosp),
    host_vs_parasite_list_leaves(datasets_alcala_cosp),
    host_vs_parasite_list_leaves(datasets_asymmetree_cosp),
    #host_vs_parasite_list_leaves(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences [High Cospeciation]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/dataset_leaves.png', dpi=300)
plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_coala_cosp),
    host_vs_parasite_list_length(datasets_treeducken_cosp),
    host_vs_parasite_list_length(datasets_alcala_cosp),
    host_vs_parasite_list_length(datasets_asymmetree_cosp),
    #host_vs_parasite_list_length(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences [High Cospeciation]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/dataset_length.png', dpi=300)
plt.show()

- compare tree metrics (cherry, sacking and coless indeces)

In [ ]:
import openpyxl
comparison_metrics_cosp = pd.concat([coala_cosp_metrics, treeducken_cosp_metrics, alcala_cosp_metrics, asymmetree_cosp_metrics])

comparison_metrics_cosp.to_excel('comparison_metrics_cosp.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index', scenario='high_cospeciation')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index', scenario='high_cospeciation')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology', scenario='high_cospeciation')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index', scenario='high_cospeciation')

Extracting and comparing reconciliation data

In [ ]:
df_box_cosp = pd.concat(
    [boxplot_coala_cosp, boxplot_treeducken_cosp, boxplot_asymmetree_cosp, boxplot_alcala_cosp],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with High Cospeciation", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_cospeciation/reconciliation.png', dpi=300)
plt.show()

### High Switch

In [ ]:
host_depths = [
    average_host_tree_depth(datasets_coala_switch),
    average_host_tree_depth(datasets_treeducken_switch),
    average_host_tree_depth(datasets_alcala_switch),
    average_host_tree_depth(datasets_asymmetree_switch),
    #average_host_tree_depth(real_dataset)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_coala_switch),
    average_parasite_tree_depth(datasets_treeducken_switch),
    average_parasite_tree_depth(datasets_alcala_switch),
    average_parasite_tree_depth(datasets_asymmetree_switch),
    #average_parasite_tree_depth(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Model with High Switch')
plt.legend()
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/avg_branches.png', dpi=300)
plt.show()

In [ ]:
host_depths = [
    host_tree_branches_list(datasets_coala_switch),
    host_tree_branches_list(datasets_treeducken_switch),
    host_tree_branches_list(datasets_alcala_switch),
    host_tree_branches_list(datasets_asymmetree_switch),
    #host_tree_branches_list(real_dataset),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_coala_switch),
    parasite_tree_branches_list(datasets_treeducken_switch),
    parasite_tree_branches_list(datasets_alcala_switch),
    parasite_tree_branches_list(datasets_asymmetree_switch),
    #parasite_tree_branches_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Datasets [Switch]")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/branches_distribution.png', dpi=300)
plt.show()

In [ ]:
host_depths = [
    host_tree_branches_list(datasets_coala_switch),
    host_tree_branches_list(datasets_treeducken_switch),
    host_tree_branches_list(datasets_alcala_switch),
    #host_tree_branches_list(datasets_asymmetree_switch),
    #host_tree_branches_list(real_dataset),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_coala_switch),
    parasite_tree_branches_list(datasets_treeducken_switch),
    parasite_tree_branches_list(datasets_alcala_switch),
    #parasite_tree_branches_list(datasets_asymmetree_switch),
    #parasite_tree_branches_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Datasets no Asymmetree [Switch]")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/branches_distribution_no_asymmetree.png', dpi=300)
plt.show()

In [ ]:
host_lengths = [
    average_host_tree_length(datasets_coala_switch),
    average_host_tree_length(datasets_treeducken_switch),
    average_host_tree_length(datasets_alcala_switch),
    average_host_tree_length(datasets_asymmetree_switch),
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_length(datasets_coala_switch),
    average_parasite_tree_length(datasets_treeducken_switch),
    average_parasite_tree_length(datasets_alcala_switch),
    average_parasite_tree_length(datasets_asymmetree_switch),
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Length')
plt.title('Average Length (max depth) of Host and Parasite Trees per Model with High Switch')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/avg_length.png', dpi=300)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
host_leaves = [
    average_host_tree_leaves(datasets_coala_switch),
    average_host_tree_leaves(datasets_treeducken_switch),
    average_host_tree_leaves(datasets_alcala_switch),
    average_host_tree_leaves(datasets_asymmetree_switch),
    #average_host_tree_leaves(real_dataset)
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_coala_switch),
    average_parasite_tree_leaves(datasets_treeducken_switch),
    average_parasite_tree_leaves(datasets_alcala_switch),
    average_parasite_tree_leaves(datasets_asymmetree_switch),
    #average_parasite_tree_leaves(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Model with High Switch')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/avg_leaves.png', dpi=300)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
host_depths = [
    host_tree_leaves_list(datasets_coala_switch),
    host_tree_leaves_list(datasets_treeducken_switch),
    host_tree_leaves_list(datasets_alcala_switch),
    host_tree_leaves_list(datasets_asymmetree_switch),
    #host_tree_leaves_list(real_dataset),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_coala_switch),
    parasite_tree_leaves_list(datasets_treeducken_switch),
    parasite_tree_leaves_list(datasets_alcala_switch),
    parasite_tree_leaves_list(datasets_asymmetree_switch),
    #parasite_tree_leaves_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets (Switch)")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/leaves_distribution.png', dpi=300)
plt.show()

In [ ]:
host_depths = [
    host_tree_leaves_list(datasets_coala_switch),
    host_tree_leaves_list(datasets_treeducken_switch),
    host_tree_leaves_list(datasets_alcala_switch),
    #host_tree_leaves_list(datasets_asymmetree_switch),
    #host_tree_leaves_list(real_dataset),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_coala_switch),
    parasite_tree_leaves_list(datasets_treeducken_switch),
    parasite_tree_leaves_list(datasets_alcala_switch),
    #parasite_tree_leaves_list(datasets_asymmetree_switch),
    #parasite_tree_leaves_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange

# Color median lines
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets no Asymmetree (Switch)")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/leaves_distribution_no_asymmetree.png', dpi=300)

plt.show()

Host vs parasite dataset comparison

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_coala_switch),
    host_vs_parasite_list(datasets_treeducken_switch),
    host_vs_parasite_list(datasets_alcala_switch),
    host_vs_parasite_list(datasets_asymmetree_switch),
    #host_vs_parasite_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences [Switch]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/dataset_branches.png', dpi=300)

plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_coala_switch),
    host_vs_parasite_list(datasets_treeducken_switch),
    host_vs_parasite_list(datasets_alcala_switch),
    #host_vs_parasite_list(datasets_asymmetree_switch),
    #host_vs_parasite_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences no Asymmetree [Switch]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/dataset_branches_no_asymmetree.png', dpi=300)
plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_coala_switch),
    host_vs_parasite_list_leaves(datasets_treeducken_switch),
    host_vs_parasite_list_leaves(datasets_alcala_switch),
    host_vs_parasite_list_leaves(datasets_asymmetree_switch),
    #host_vs_parasite_list_leaves(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences [Switch]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/leaves_distribution.png', dpi=300)

plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_coala_switch),
    host_vs_parasite_list_leaves(datasets_treeducken_switch),
    host_vs_parasite_list_leaves(datasets_alcala_switch),
    #host_vs_parasite_list_leaves(datasets_asymmetree_switch),
    #host_vs_parasite_list_leaves(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Leaves (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences no Asymmetree [Switch]")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/leaves_distribution_no_asymmetree.png', dpi=300)
plt.show()

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_coala_switch),
    host_vs_parasite_list_length(datasets_treeducken_switch),
    host_vs_parasite_list_length(datasets_alcala_switch),
    host_vs_parasite_list_length(datasets_asymmetree_switch),
    #host_vs_parasite_list_length(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree length (max depth)")
plt.title("Distribution of Host–Parasite Tree Length Differences (Switch)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/length_distribution.png', dpi=300)
plt.show()

In [ ]:
comparison_metrics_switch = pd.concat([coala_switch_metrics, treeducken_switch_metrics, alcala_switch_metrics, asymmetree_switch_metrics])

comparison_metrics_switch.to_excel('comparison_metrics_switch.xlsx', index=False)
    
metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite'
]
    

plot_metric_comparison_paired(comparison_metrics_switch, 'cherry_index', scenario='high_switch')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index', scenario='high_switch')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index_topology', scenario='high_switch')
plot_metric_comparison_paired(comparison_metrics_switch, 'colless_index', scenario='high_switch')

Checking reconciliation on the different models

In [ ]:
# Combine all models into one DataFrame
df_box_switch = pd.concat(
    [boxplot_coala_switch, boxplot_treeducken_switch, boxplot_asymmetree_switch, boxplot_alcala_switch],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_switch = df_box_switch.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_switch = df_box_switch.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_switch


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_switch,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with high Host Switch", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/high_switch/reconciliation.png', dpi=300)
plt.show()

### Medium Values

In [ ]:
host_depths = [
    average_host_tree_depth(datasets_coala_med),
    average_host_tree_depth(datasets_treeducken_medium),
    average_host_tree_depth(datasets_alcala_medium),
    average_host_tree_depth(datasets_asymmetree_medium),
    #average_host_tree_depth(real_dataset)
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_coala_med),
    average_parasite_tree_depth(datasets_treeducken_medium),
    average_parasite_tree_depth(datasets_alcala_medium),
    average_parasite_tree_depth(datasets_asymmetree_medium),
    #average_parasite_tree_depth(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Model with Medium Values')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/avg_branches.png', dpi=300)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
host_depths = [
    host_tree_branches_list(datasets_coala_med),
    host_tree_branches_list(datasets_treeducken_medium),
    host_tree_branches_list(datasets_alcala_medium),
    host_tree_branches_list(datasets_asymmetree_medium),
    #host_tree_branches_list(real_dataset),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_coala_med),
    parasite_tree_branches_list(datasets_treeducken_medium),
    parasite_tree_branches_list(datasets_alcala_medium),
    parasite_tree_branches_list(datasets_asymmetree_medium),
    #parasite_tree_branches_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Datasets (Medium)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/branches_distribution.png', dpi=300)

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
host_lengths = [
    average_host_tree_length(datasets_coala_med),
    average_host_tree_length(datasets_treeducken_medium),
    average_host_tree_length(datasets_alcala_medium),
    average_host_tree_length(datasets_asymmetree_medium),
]

# Compute average lengths
parasite_lengths = [
    average_parasite_tree_length(datasets_coala_med),
    average_parasite_tree_length(datasets_treeducken_medium),
    average_parasite_tree_length(datasets_alcala_medium),
    average_parasite_tree_length(datasets_asymmetree_medium),
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_lengths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_lengths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Length')
plt.title('Average Length (max depth) of Host and Parasite Trees per Model with Medium Values')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/avg_length.png', dpi=300)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
host_leaves = [
    average_host_tree_leaves(datasets_coala_med),
    average_host_tree_leaves(datasets_treeducken_medium),
    average_host_tree_leaves(datasets_alcala_medium),
    average_host_tree_leaves(datasets_asymmetree_medium),
    #average_host_tree_leaves(real_dataset)
]

# Compute average lengths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_coala_med),
    average_parasite_tree_leaves(datasets_treeducken_medium),
    average_parasite_tree_leaves(datasets_alcala_medium),
    average_parasite_tree_leaves(datasets_asymmetree_medium),
    #average_parasite_tree_leaves(real_dataset)
]

# Plotting
labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree # Leaves')
plt.title('Average # Leaves of Host and Parasite Trees per Model with Medium Values')
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/avg_leaves.png', dpi=300)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
host_depths = [
    host_tree_leaves_list(datasets_coala_med),
    host_tree_leaves_list(datasets_treeducken_medium),
    host_tree_leaves_list(datasets_alcala_medium),
    host_tree_leaves_list(datasets_asymmetree_medium),
    #host_tree_leaves_list(real_dataset),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_coala_med),
    parasite_tree_leaves_list(datasets_treeducken_medium),
    parasite_tree_leaves_list(datasets_alcala_medium),
    parasite_tree_leaves_list(datasets_asymmetree_medium),
    #parasite_tree_leaves_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets (Medium)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/leaves_distribution.png', dpi=300)


# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

host vs parasite metrics in dataset

In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list(datasets_coala_med),
    host_vs_parasite_list(datasets_treeducken_medium),
    host_vs_parasite_list(datasets_alcala_medium),
    host_vs_parasite_list(datasets_asymmetree_medium),
    #host_vs_parasite_list(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences (Medium)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/dataset_branches.png', dpi=300)
plt.show()


In [ ]:

# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_leaves(datasets_coala_med),
    host_vs_parasite_list_leaves(datasets_treeducken_medium),
    host_vs_parasite_list_leaves(datasets_alcala_medium),
    host_vs_parasite_list_leaves(datasets_asymmetree_medium),
    #host_vs_parasite_list_leaves(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num leaves)")
plt.title("Distribution of Host–Parasite Tree Leaves Differences (Medium)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/dataset_leaves.png', dpi=300)
plt.show()


In [ ]:
# Host vs pparasite dataset comparison
diffs = [
    host_vs_parasite_list_length(datasets_coala_med),
    host_vs_parasite_list_length(datasets_treeducken_medium),
    host_vs_parasite_list_length(datasets_alcala_medium),
    host_vs_parasite_list_length(datasets_asymmetree_medium),
    #host_vs_parasite_list_length(real_dataset),
]

labels = ['Coala', 'Treeducken', 'Cophylo', 'Asymmetree']

plt.figure(figsize=(10,6))
plt.boxplot(diffs, labels=labels, patch_artist=True)
plt.ylabel("Host - Parasite Tree Depth (num branches)")
plt.title("Distribution of Host–Parasite Tree Depth Differences (Medium)")
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/dataset_length.png', dpi=300)
plt.show()

In [ ]:
comparison_metrics_medium = pd.concat([coala_medium_metrics, treeducken_medium_metrics, alcala_medium_metrics, asymmetree_medium_metrics])

comparison_metrics_medium.to_excel('comparison_metrics_medium.xlsx', index=False)

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite'
]



plot_metric_comparison_paired(comparison_metrics_medium, 'cherry_index', scenario='medium')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index', scenario='medium')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index_topology', scenario='medium')
plot_metric_comparison_paired(comparison_metrics_medium, 'colless_index', scenario='medium')

Comparing reconciliation statistics

In [ ]:
df_box_med = pd.concat(
    [boxplot_coala_med, boxplot_treeducken_medium, boxplot_asymmetree_medium, boxplot_alcala_med],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_med = df_box_med.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_med = df_box_med.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_med


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_med,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with Medium values", fontsize=14, pad=15)
plt.savefig('/Users/gabriele/synthetic_cophylo/comparison_trees/medium/reconciliation.png', dpi=300)

plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Compare Networks

### High Cospeciation

In [ ]:
treeducken_graphs_cosp = generate_all_bipartite_graphs(datasets_treeducken_cosp, show_plots=False, save_dir="treeducken/treeducken_cosp_bipartite_graphs")
coala_graphs_cosp = generate_all_bipartite_graphs(datasets_coala_cosp, show_plots=False, save_dir="coala/coala_cosp_bipartite_graphs")
alcala_graphs_cosp = generate_all_bipartite_graphs(datasets_alcala_cosp, show_plots=False, save_dir="alcala/alcala_cosp_bipartite_graphs")
asymmetree_graphs_cosp = generate_all_bipartite_graphs(datasets_asymmetree_cosp, show_plots=False, save_dir="asymmetree/asymmetree_cosp_bipartite_graphs")
real_data_graphs = generate_all_bipartite_graphs(real_dataset, show_plots=False, save_dir="real/real_bipartite_graphs")

In [ ]:

results = analyze_model_graphs(treeducken_graphs_cosp, "Treeducken") + \
          analyze_model_graphs(coala_graphs_cosp, "Coala") + \
          analyze_model_graphs(alcala_graphs_cosp, "Cophylo") + \
          analyze_model_graphs(asymmetree_graphs_cosp, "Asymmetree") + \
          analyze_model_graphs(real_data_graphs, "Real")

df_graph_metrics_cosp = pd.DataFrame(results)
df_graph_metrics_cosp["model"] = df_graph_metrics_cosp["dataset"].apply(lambda x: x.split("_")[0])

df_graph_metrics_cosp.to_excel("bipartite_graph_metrics_cosp.xlsx", index=False)
df_graph_metrics_cosp

In [ ]:
df_graph_metrics_cosp["model"] = df_graph_metrics_cosp["dataset"].apply(lambda x: x.split("_")[0])

metrics_to_plot = [
    "density", 
    "avg_host_degree", 
    "avg_parasite_degree", 
    "host_hotspots", 
    "degree_assortativity", 
    "num_components"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_graph_metrics_cosp, x="model", y=metric, hue='model')
    plt.title(f"{metric.replace('_', ' ').title()} per Model")
    plt.ylabel(metric.replace('_', ' ').title())
    plt.xlabel("Model")
    plt.tight_layout()
    plt.show()

### High Switch

In [ ]:
treeducken_graphs_switch = generate_all_bipartite_graphs(datasets_treeducken_switch, show_plots=False, save_dir="treeducken/treeducken_switch_bipartite_graphs")
coala_graphs_switch = generate_all_bipartite_graphs(datasets_coala_switch, show_plots=False, save_dir="coala/coala_switch_bipartite_graphs")
alcala_graphs_switch = generate_all_bipartite_graphs(datasets_alcala_switch, show_plots=False, save_dir="Cophylo/alcala_switch_bipartite_graphs")
asymmetree_graphs_switch = generate_all_bipartite_graphs(datasets_asymmetree_switch, show_plots=False, save_dir="asymmetree/asymmetree_switch_bipartite_graphs")

In [ ]:

results = analyze_model_graphs(treeducken_graphs_switch, "Treeducken") + \
          analyze_model_graphs(coala_graphs_switch, "Coala") + \
          analyze_model_graphs(alcala_graphs_switch, "Cophylo") + \
          analyze_model_graphs(asymmetree_graphs_switch, "Asymmetree") + \
          analyze_model_graphs(real_data_graphs, "Real")

df_graph_metrics_switch = pd.DataFrame(results)
df_graph_metrics_switch["model"] = df_graph_metrics_switch["dataset"].apply(lambda x: x.split("_")[0])

df_graph_metrics_switch.to_excel("bipartite_graph_metrics_switch.xlsx", index=False)
df_graph_metrics_switch

In [ ]:
df_graph_metrics_switch["model"] = df_graph_metrics_switch["dataset"].apply(lambda x: x.split("_")[0])

metrics_to_plot = [
    "density", 
    "avg_host_degree", 
    "avg_parasite_degree", 
    "host_hotspots", 
    "degree_assortativity", 
    "num_components"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_graph_metrics_switch, x="model", y=metric, hue='model')
    plt.title(f"{metric.replace('_', ' ').title()} per Model")
    plt.ylabel(metric.replace('_', ' ').title())
    plt.xlabel("Model")
    plt.tight_layout()
    plt.show()

### Medium values

In [ ]:
treeducken_graphs_medium = generate_all_bipartite_graphs(datasets_treeducken_medium, show_plots=False, save_dir="treeducken/treeducken_medium_bipartite_graphs")
coala_graphs_medium = generate_all_bipartite_graphs(datasets_coala_med, show_plots=False, save_dir="coala/coala_medium_bipartite_graphs")
alcala_graphs_medium = generate_all_bipartite_graphs(datasets_alcala_medium, show_plots=False, save_dir="Cophylo/alcala_medium_bipartite_graphs")
asymmetree_graphs_medium = generate_all_bipartite_graphs(datasets_asymmetree_medium, show_plots=False, save_dir="asymmetree/asymmetree_medium_bipartite_graphs")

In [ ]:
results = analyze_model_graphs(treeducken_graphs_medium, "Treeducken") + \
          analyze_model_graphs(coala_graphs_medium, "Coala") + \
          analyze_model_graphs(alcala_graphs_medium, "Cophylo") + \
          analyze_model_graphs(asymmetree_graphs_medium, "Asymmetree") + \
          analyze_model_graphs(real_data_graphs, "Real")

df_graph_metrics_medium = pd.DataFrame(results)
df_graph_metrics_medium["model"] = df_graph_metrics_medium["dataset"].apply(lambda x: x.split("_")[0])
df_graph_metrics_medium.to_excel("bipartite_graph_metrics_medium.xlsx", index=False)
df_graph_metrics_medium

In [ ]:
df_graph_metrics_medium["model"] = df_graph_metrics_medium["dataset"].apply(lambda x: x.split("_")[0])

metrics_to_plot = [
    "density", 
    "avg_host_degree", 
    "avg_parasite_degree", 
    "host_hotspots", 
    "degree_assortativity", 
    "num_components"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df_graph_metrics_medium, x="model", y=metric, hue="model")
    plt.ylabel(metric.replace('_', ' ').title())
    plt.xlabel("Model")
    plt.tight_layout()
    plt.show()

# Real Data network analysis

Find hotspots in real data

In [ ]:
real_networks = analyze_model_graphs(real_data_graphs, "Real")
df_real_networks = pd.DataFrame(real_networks)
df_real_networks

In [ ]:
records = []   # this will become the DataFrame

for k, g in real_data_graphs.items():
    hosts, parasites = get_node_sets(g)
    degrees = dict(g.degree())

    host_degrees = [degrees[n] for n in hosts]
    mean = np.mean(host_degrees)
    std  = np.std(host_degrees)
    threshold = mean + std

    # hotspot hosts
    host_hotspots = [n for n in hosts if degrees[n] > threshold]

    for host in host_hotspots:
        host_degree = degrees[host]

        # list parasite neighbors only
        parasite_neighbors = [
            nbr for nbr in g.neighbors(host) if nbr in parasites
        ]

        records.append({
            "dataset": k,
            "host": host,
            "host_degree": host_degree,
            "mean_host_degree": mean,
            "std_host_degree": std,
            "threshold": threshold,
            "parasite_neighbors": parasite_neighbors,
            "num_parasite_neighbors": len(parasite_neighbors)
        })

# Convert to DataFrame
df = pd.DataFrame(records)
df.to_excel("real_host_hotspots.xlsx", index=False)
df

# Compare time in treeducken and asymmetree

## Treeducken

In [ ]:
import os

### high cosp

In [ ]:
datasets_treeducken_cosp_norm= parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highCosp/Datasets')
datasets_treeducken_cosp_small = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/small_time/high_cosp/Datasets')
datasets_treeducken_cosp_large = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/big_time/high_cosp/Datasets')
for dataset in datasets_treeducken_cosp_norm:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_cosp_small:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_cosp_large:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    


In [ ]:
import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_treeducken_cosp_small),
    average_host_tree_depth(datasets_treeducken_cosp_norm),
    average_host_tree_depth(datasets_treeducken_cosp_large),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_treeducken_cosp_small),
    average_parasite_tree_depth(datasets_treeducken_cosp_norm),
    average_parasite_tree_depth(datasets_treeducken_cosp_large),
    
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Treeducken Model with High Cosp')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_treeducken_cosp_small),
    host_tree_branches_list(datasets_treeducken_cosp_norm),
    host_tree_branches_list(datasets_treeducken_cosp_large),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_treeducken_cosp_small),
    parasite_tree_branches_list(datasets_treeducken_cosp_norm),
    parasite_tree_branches_list(datasets_treeducken_cosp_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Depth (# branches)")
plt.title("Distribution of Tree Depths Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_treeducken_cosp_small),
    average_host_tree_leaves(datasets_treeducken_cosp_norm),
    average_host_tree_leaves(datasets_treeducken_cosp_large),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_treeducken_cosp_small),
    average_parasite_tree_leaves(datasets_treeducken_cosp_norm),
    average_parasite_tree_leaves(datasets_treeducken_cosp_large),
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Treeducken Model with High Cosp')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_leaves_list(datasets_treeducken_cosp_small),
    host_tree_leaves_list(datasets_treeducken_cosp_norm),
    host_tree_leaves_list(datasets_treeducken_cosp_large),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_treeducken_cosp_small),
    parasite_tree_leaves_list(datasets_treeducken_cosp_norm),
    parasite_tree_leaves_list(datasets_treeducken_cosp_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
import openpyxl
treeducken_cosp_small = compute_tree_metrics(datasets_treeducken_cosp_small, model_name="Treeducken_Cosp_Small")
treeducken_cosp_normal = compute_tree_metrics(datasets_treeducken_cosp_norm, model_name="Treeducken_Cosp_Normal")
treeducken_cosp_large = compute_tree_metrics(datasets_treeducken_cosp_large, model_name="Treeducken_Cosp_Large")

comparison_metrics_cosp = pd.concat([treeducken_cosp_small, treeducken_cosp_normal, treeducken_cosp_large])


metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_cosp, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_cosp, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_cosp, 'colless_index')


In [ ]:
recon_cosp_treeducken_small = extract_sim_stats(datasets_treeducken_cosp_small)
recon_cosp_treeducken_norm = extract_sim_stats(datasets_treeducken_cosp_norm)
recon_cosp_treeducken_large = extract_sim_stats(datasets_treeducken_cosp_large)

flattened_cosp_treeducken_small = flatten_sim_stats(recon_cosp_treeducken_small).fillna(0)
flattened_cosp_treeducken_norm = flatten_sim_stats(recon_cosp_treeducken_norm).fillna(0)
flattened_cosp_treeducken_large = flatten_sim_stats(recon_cosp_treeducken_large).fillna(0)

boxplot_treeducken_cosp_small = flattened_cosp_treeducken_small[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_cosp_small.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)

boxplot_treeducken_cosp_norm = flattened_cosp_treeducken_norm[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_cosp_norm.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)

boxplot_treeducken_cosp_large = flattened_cosp_treeducken_large[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_cosp_large.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_cosp_small["Model"] = "Treeducken_Small"
boxplot_treeducken_cosp_norm["Model"] = "Treeducken_Normal"
boxplot_treeducken_cosp_large["Model"] = "Treeducken_Large"


# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_treeducken_cosp_small, boxplot_treeducken_cosp_norm, boxplot_treeducken_cosp_large],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])

df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with High Cospeciation", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### high switches

In [ ]:
datasets_treeducken_switch_norm= parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highSwitch/Datasets')
datasets_treeducken_switch_small = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/small_time/high_switch/Datasets')
datasets_treeducken_switch_large = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/big_time/high_switch/Datasets')
for dataset in datasets_treeducken_switch_norm:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_switch_small:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_switch_large:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    


In [ ]:
import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_treeducken_switch_small),
    average_host_tree_depth(datasets_treeducken_switch_norm),
    average_host_tree_depth(datasets_treeducken_switch_large),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_treeducken_switch_small),
    average_parasite_tree_depth(datasets_treeducken_switch_norm),
    average_parasite_tree_depth(datasets_treeducken_switch_large),
    
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Treeducken Model with High Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_treeducken_switch_small),
    host_tree_branches_list(datasets_treeducken_switch_norm),
    host_tree_branches_list(datasets_treeducken_switch_large),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_treeducken_switch_small),
    parasite_tree_branches_list(datasets_treeducken_switch_norm),
    parasite_tree_branches_list(datasets_treeducken_switch_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# branches)")
plt.title("Distribution of Tree Branches Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:

#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_treeducken_switch_small),
    average_host_tree_leaves(datasets_treeducken_switch_norm),
    average_host_tree_leaves(datasets_treeducken_switch_large),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_treeducken_switch_small),
    average_parasite_tree_leaves(datasets_treeducken_switch_norm),
    average_parasite_tree_leaves(datasets_treeducken_switch_large),
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Treeducken Model with High Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_leaves_list(datasets_treeducken_switch_small),
    host_tree_leaves_list(datasets_treeducken_switch_norm),
    host_tree_leaves_list(datasets_treeducken_switch_large),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_treeducken_switch_small),
    parasite_tree_leaves_list(datasets_treeducken_switch_norm),
    parasite_tree_leaves_list(datasets_treeducken_switch_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:

import openpyxl
treeducken_switch_small = compute_tree_metrics(datasets_treeducken_switch_small, model_name="Treeducken_Switch_Small")
treeducken_switch_normal = compute_tree_metrics(datasets_treeducken_switch_norm, model_name="Treeducken_Switch_Normal")
treeducken_switch_large = compute_tree_metrics(datasets_treeducken_switch_large, model_name="Treeducken_Switch_Large")

comparison_metrics_switch = pd.concat([treeducken_switch_small, treeducken_switch_normal, treeducken_switch_large])

metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_switch, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_switch, 'colless_index')


In [ ]:

recon_switch_treeducken_small = extract_sim_stats(datasets_treeducken_switch_small)
recon_switch_treeducken_norm = extract_sim_stats(datasets_treeducken_switch_norm)
recon_switch_treeducken_large = extract_sim_stats(datasets_treeducken_switch_large)

flattened_switch_treeducken_small = flatten_sim_stats(recon_switch_treeducken_small).fillna(0)
flattened_switch_treeducken_norm = flatten_sim_stats(recon_switch_treeducken_norm).fillna(0)
flattened_switch_treeducken_large = flatten_sim_stats(recon_switch_treeducken_large).fillna(0)

boxplot_treeducken_switch_small = flattened_switch_treeducken_small[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_switch_small.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)

boxplot_treeducken_switch_norm = flattened_switch_treeducken_norm[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_switch_norm.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)
boxplot_treeducken_switch_large = flattened_switch_treeducken_large[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_switch_large.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_switch_small["Model"] = "Treeducken_Small"
boxplot_treeducken_switch_norm["Model"] = "Treeducken_Normal"
boxplot_treeducken_switch_large["Model"] = "Treeducken_Large"


# Combine all models into one DataFrame
df_box_switch = pd.concat(
    [boxplot_treeducken_switch_small, boxplot_treeducken_switch_norm, boxplot_treeducken_switch_large],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_switch = df_box_switch.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_switch = df_box_switch.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_switch


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_switch,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with High Switch", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### Medium scenario


In [ ]:
datasets_treeducken_medium_norm= parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_medium/Datasets')
datasets_treeducken_medium_small = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/small_time/medium/Datasets')
datasets_treeducken_medium_large = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/time_analysis/big_time/medium/Datasets')
for dataset in datasets_treeducken_medium_norm:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_medium_small:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    
for dataset in datasets_treeducken_medium_large:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])
    dataset['host_tree']['num_leaves'] -= dataset['sim_stats'].get('Host_Extinctions') or 0
    dataset['parasite_tree']['num_leaves'] -= dataset['sim_stats'].get('Symbiont_Extinctions') or 0
    


In [ ]:
import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_treeducken_medium_small),
    average_host_tree_depth(datasets_treeducken_medium_norm),
    average_host_tree_depth(datasets_treeducken_medium_large),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_treeducken_medium_small),
    average_parasite_tree_depth(datasets_treeducken_medium_norm),
    average_parasite_tree_depth(datasets_treeducken_medium_large),
    
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Treeducken Model Medium')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_branches_list(datasets_treeducken_medium_small),
    host_tree_branches_list(datasets_treeducken_medium_norm),
    host_tree_branches_list(datasets_treeducken_medium_large),
]

parasite_depths = [
    parasite_tree_branches_list(datasets_treeducken_medium_small),
    parasite_tree_branches_list(datasets_treeducken_medium_norm),
    parasite_tree_branches_list(datasets_treeducken_medium_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Branches (# branches)")
plt.title("Distribution of Tree Branches Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:

#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_treeducken_medium_small),
    average_host_tree_leaves(datasets_treeducken_medium_norm),
    average_host_tree_leaves(datasets_treeducken_medium_large),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_treeducken_medium_small),
    average_parasite_tree_leaves(datasets_treeducken_medium_norm),
    average_parasite_tree_leaves(datasets_treeducken_medium_large),
]

# Plotting
labels = ['Treeducken 1', 'Treeducken 2',  'Treeducken 3']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Treeducken Model Medium')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Collect LISTS (not averages!)
host_depths = [
    host_tree_leaves_list(datasets_treeducken_medium_small),
    host_tree_leaves_list(datasets_treeducken_medium_norm),
    host_tree_leaves_list(datasets_treeducken_medium_large),
]

parasite_depths = [
    parasite_tree_leaves_list(datasets_treeducken_medium_small),
    parasite_tree_leaves_list(datasets_treeducken_medium_norm),
    parasite_tree_leaves_list(datasets_treeducken_medium_large),
]

labels = ['Treeducken 1', 'Treeducken 2', 'Treeducken 3']

plt.figure(figsize=(12, 6))

# Host trees boxplot (left)
host_bp = plt.boxplot(
    host_depths,
    positions=[i*2 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Parasite trees boxplot (right)
parasite_bp = plt.boxplot(
    parasite_depths,
    positions=[i*2 + 1 for i in range(len(labels))],
    widths=0.6,
    patch_artist=True
)

# Color the boxes
for box in host_bp['boxes']:
    box.set(facecolor='cornflowerblue')  # blue

for box in parasite_bp['boxes']:
    box.set(facecolor='orange')          # orange
    
for median in host_bp['medians']:
    median.set(color='black', linewidth=2)

for median in parasite_bp['medians']:
    median.set(color='black', linewidth=2)

# x-ticks centered between pairs
plt.xticks([i*2 + 0.5 for i in range(len(labels))], labels)

plt.ylabel("Tree Leaves (# leaves)")
plt.title("Distribution of Tree Leaves Across Datasets")

# Legend handles
from matplotlib.patches import Patch
plt.legend(
    handles=[
        Patch(facecolor='cornflowerblue', label='Host'),
        Patch(facecolor='orange', label='Parasite')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()

In [ ]:
import openpyxl
treeducken_medium_small = compute_tree_metrics(datasets_treeducken_medium_small, model_name="Treeducken_Switch_Small")
treeducken_medium_norm = compute_tree_metrics(datasets_treeducken_medium_norm, model_name="Treeducken_Switch_Normal")
treeducken_medium_large = compute_tree_metrics(datasets_treeducken_medium_large, model_name="Treeducken_Switch_Large")

comparison_metrics_switch = pd.concat([treeducken_medium_small, treeducken_medium_norm, treeducken_medium_large])
metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_switch, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_switch, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_switch, 'colless_index')

In [ ]:
recon_medium_treeducken_small = extract_sim_stats(datasets_treeducken_medium_small)
recon_medium_treeducken_norm = extract_sim_stats(datasets_treeducken_medium_norm)
recon_medium_treeducken_large = extract_sim_stats(datasets_treeducken_medium_large)

flattened_medium_treeducken_small = flatten_sim_stats(recon_medium_treeducken_small).fillna(0)
flattened_medium_treeducken_norm = flatten_sim_stats(recon_medium_treeducken_norm).fillna(0)
flattened_medium_treeducken_large = flatten_sim_stats(recon_medium_treeducken_large).fillna(0)

boxplot_treeducken_medium_small = flattened_medium_treeducken_small[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_medium_small.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)

boxplot_treeducken_medium_norm = flattened_medium_treeducken_norm[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_medium_norm.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)
boxplot_treeducken_medium_large = flattened_medium_treeducken_large[['filename', 'Cospeciations', 'Host_Spread_Switches']]
boxplot_treeducken_medium_large.rename(columns={'Cospeciations': 'Cospeciation', 'Host_Spread_Switches': 'Host_switch'}, inplace=True)


boxplot_treeducken_medium_small["Model"] = "Treeducken_Small"
boxplot_treeducken_medium_norm["Model"] = "Treeducken_Normal"
boxplot_treeducken_medium_large["Model"] = "Treeducken_Large"


# Combine all models into one DataFrame
df_box_medium= pd.concat(
    [boxplot_treeducken_medium_small, boxplot_treeducken_medium_norm, boxplot_treeducken_medium_large],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_medium = df_box_medium.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_medium = df_box_medium.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_medium


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_medium,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models Medium", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Asymmetree

### Low Switch

In [ ]:
datasets_asymmetree_cosp_normal = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_low_switch/Datasets')
datasets_asymmetree_cosp_smaller = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_low_switch_smaller/Datasets')
for dataset in datasets_asymmetree_cosp_normal:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])

for dataset in datasets_asymmetree_cosp_smaller:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])


In [ ]:

import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_asymmetree_cosp_smaller),
    average_host_tree_depth(datasets_asymmetree_cosp_normal),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_asymmetree_cosp_smaller),
    average_parasite_tree_depth(datasets_asymmetree_cosp_normal),
    
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Asymmetree Model Low Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_asymmetree_cosp_smaller),
    average_host_tree_leaves(datasets_asymmetree_cosp_normal),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_asymmetree_cosp_smaller),
    average_parasite_tree_leaves(datasets_asymmetree_cosp_normal),
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Asymmetree Model Low Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

import openpyxl
asymmetree_cosp_small = compute_tree_metrics(datasets_asymmetree_cosp_smaller, model_name="Asymmetree_Cosp_Small")
asymmetree_cosp_normal = compute_tree_metrics(datasets_asymmetree_cosp_normal, model_name="Asymmetree_Cosp_Normal")

comparison_metrics_medium = pd.concat([asymmetree_cosp_small, asymmetree_cosp_normal])
metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_medium, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_medium, 'colless_index')


In [ ]:


recon_medium_asymmetree_cosp_small = extract_sim_stats(datasets_asymmetree_cosp_smaller)
recon_medium_asymmetree_cosp_norm = extract_sim_stats(datasets_asymmetree_cosp_normal)

flattened_cosp_asymmetree_small = flatten_sim_stats(recon_medium_asymmetree_cosp_small).fillna(0)
flattened_cosp_asymmetree_norm = flatten_sim_stats(recon_medium_asymmetree_cosp_norm).fillna(0)


flattened_cosp_asymmetree_small['Total Events'] = (flattened_cosp_asymmetree_small['Horizontal_Gene_Transfer'] +
    flattened_cosp_asymmetree_small['Duplication'] +
    flattened_cosp_asymmetree_small['Speciation'] +
    flattened_cosp_asymmetree_small['Loss'])
flattened_cosp_asymmetree_small['Horizontal_Gene_Transfer_Rate'] = flattened_cosp_asymmetree_small['Horizontal_Gene_Transfer'] / flattened_cosp_asymmetree_small['Total Events']
flattened_cosp_asymmetree_small['Duplication_Rate'] = flattened_cosp_asymmetree_small['Duplication'] / flattened_cosp_asymmetree_small['Total Events']
flattened_cosp_asymmetree_small['Speciation_Rate'] = flattened_cosp_asymmetree_small['Speciation'] / flattened_cosp_asymmetree_small['Total Events']
flattened_cosp_asymmetree_small['Loss_Rate'] = flattened_cosp_asymmetree_small['Loss'] / flattened_cosp_asymmetree_small['Total Events']
flattened_cosp_asymmetree_small.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

flattened_cosp_asymmetree_norm['Total Events'] = (flattened_cosp_asymmetree_norm['Horizontal_Gene_Transfer'] +
    flattened_cosp_asymmetree_norm['Duplication'] +
    flattened_cosp_asymmetree_norm['Speciation'] +
    flattened_cosp_asymmetree_norm['Loss'])
flattened_cosp_asymmetree_norm['Horizontal_Gene_Transfer_Rate'] = flattened_cosp_asymmetree_norm['Horizontal_Gene_Transfer'] / flattened_cosp_asymmetree_norm['Total Events']
flattened_cosp_asymmetree_norm['Duplication_Rate'] = flattened_cosp_asymmetree_norm['Duplication'] / flattened_cosp_asymmetree_norm['Total Events']
flattened_cosp_asymmetree_norm['Speciation_Rate'] = flattened_cosp_asymmetree_norm['Speciation'] / flattened_cosp_asymmetree_norm['Total Events']
flattened_cosp_asymmetree_norm['Loss_Rate'] = flattened_cosp_asymmetree_norm['Loss'] / flattened_cosp_asymmetree_norm['Total Events']
flattened_cosp_asymmetree_norm.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_cosp_small = flattened_cosp_asymmetree_small[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_cosp_small.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)
boxplot_asymmetree_cosp_norm = flattened_cosp_asymmetree_norm[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_cosp_norm.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)



boxplot_asymmetree_cosp_small["Model"] = "AsymmeTree_small"
boxplot_asymmetree_cosp_norm["Model"] = "AsymmeTree_norm"

# Combine all models into one DataFrame
df_box_cosp = pd.concat(
    [boxplot_asymmetree_cosp_small, boxplot_asymmetree_cosp_norm],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_cosp = df_box_cosp.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_cosp = df_box_cosp.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_cosp



In [ ]:


import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_cosp,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with Low Switch", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### High switch

In [ ]:
datasets_asymmetree_switch_normal = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_high_switch/Datasets')
datasets_asymmetree_switch_smaller = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_high_switch_smaller/Datasets')
for dataset in datasets_asymmetree_switch_normal:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])

for dataset in datasets_asymmetree_switch_smaller:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])


In [ ]:

import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_asymmetree_switch_smaller),
    average_host_tree_depth(datasets_asymmetree_switch_normal),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_asymmetree_switch_smaller),
    average_parasite_tree_depth(datasets_asymmetree_switch_normal),
    
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Asymmetree Model High Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_asymmetree_switch_smaller),
    average_host_tree_leaves(datasets_asymmetree_switch_normal),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_asymmetree_switch_smaller),
    average_parasite_tree_leaves(datasets_asymmetree_switch_normal),
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Asymmetree Model High Switch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

import openpyxl
asymmetree_switch_small = compute_tree_metrics(datasets_asymmetree_switch_smaller, model_name="Asymmetree_Switch_Small")
asymmetree_switch_normal = compute_tree_metrics(datasets_asymmetree_switch_normal, model_name="Asymmetree_Switch_Normal")

comparison_metrics_medium = pd.concat([asymmetree_switch_small, asymmetree_switch_normal])
metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_medium, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_medium, 'colless_index')


In [ ]:
recon_medium_asymmetree_switch_small = extract_sim_stats(datasets_asymmetree_switch_smaller)
recon_medium_asymmetree_switch_norm = extract_sim_stats(datasets_asymmetree_switch_normal)

flattened_switch_asymmetree_small = flatten_sim_stats(recon_medium_asymmetree_switch_small).fillna(0)
flattened_switch_asymmetree_norm = flatten_sim_stats(recon_medium_asymmetree_switch_norm).fillna(0)

flattened_switch_asymmetree_small['Total Events'] = (flattened_switch_asymmetree_small['Horizontal_Gene_Transfer'] +
    flattened_switch_asymmetree_small['Duplication'] +
    flattened_switch_asymmetree_small['Speciation'] +
    flattened_switch_asymmetree_small['Loss'])
flattened_switch_asymmetree_small['Horizontal_Gene_Transfer_Rate'] = flattened_switch_asymmetree_small['Horizontal_Gene_Transfer'] / flattened_switch_asymmetree_small['Total Events']
flattened_switch_asymmetree_small['Duplication_Rate'] = flattened_switch_asymmetree_small['Duplication'] / flattened_switch_asymmetree_small['Total Events']
flattened_switch_asymmetree_small['Speciation_Rate'] = flattened_switch_asymmetree_small['Speciation'] / flattened_switch_asymmetree_small['Total Events']
flattened_switch_asymmetree_small['Loss_Rate'] = flattened_switch_asymmetree_small['Loss'] / flattened_switch_asymmetree_small['Total Events']
flattened_switch_asymmetree_small.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

flattened_switch_asymmetree_norm['Total Events'] = (flattened_switch_asymmetree_norm['Horizontal_Gene_Transfer'] +
    flattened_switch_asymmetree_norm['Duplication'] +
    flattened_switch_asymmetree_norm['Speciation'] +
    flattened_switch_asymmetree_norm['Loss'])
flattened_switch_asymmetree_norm['Horizontal_Gene_Transfer_Rate'] = flattened_switch_asymmetree_norm['Horizontal_Gene_Transfer'] / flattened_switch_asymmetree_norm['Total Events']
flattened_switch_asymmetree_norm['Duplication_Rate'] = flattened_switch_asymmetree_norm['Duplication'] / flattened_switch_asymmetree_norm['Total Events']
flattened_switch_asymmetree_norm['Speciation_Rate'] = flattened_switch_asymmetree_norm['Speciation'] / flattened_switch_asymmetree_norm['Total Events']
flattened_switch_asymmetree_norm['Loss_Rate'] = flattened_switch_asymmetree_norm['Loss'] / flattened_switch_asymmetree_norm['Total Events']
flattened_switch_asymmetree_norm.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_switch_small = flattened_switch_asymmetree_small[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_switch_small.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)
boxplot_asymmetree_switch_norm = flattened_switch_asymmetree_norm[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_switch_norm.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)


boxplot_asymmetree_switch_small["Model"] = "AsymmeTree_small"
boxplot_asymmetree_switch_norm["Model"] = "AsymmeTree_norm"

# Combine all models into one DataFrame
df_box_switch = pd.concat(
    [boxplot_asymmetree_switch_small, boxplot_asymmetree_switch_norm],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_switch = df_box_switch.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_switch = df_box_switch.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_switch


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_switch,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models with High Switch", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### Medium

In [ ]:
datasets_asymmetree_medium_normal = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_medium/Datasets')
datasets_asymmetree_medium_smaller = parse_all_asymmetree_in_folder('/Users/gabriele/synthetic_cophylo/generate_asymmetree/generated_trees_medium_smaller/Datasets')
for dataset in datasets_asymmetree_medium_normal:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])

for dataset in datasets_asymmetree_medium_smaller:
    dataset['host_tree'] = load_tree(dataset['host_newick'])
    dataset['parasite_tree'] = load_tree(dataset['parasite_newick'])
    dataset['host_tree_stats'] = get_tree_stats(dataset['host_tree'])
    dataset['parasite_tree_stats'] = get_tree_stats(dataset['parasite_tree'])


In [ ]:

import matplotlib.pyplot as plt

# Compute average depths
host_depths = [
    average_host_tree_depth(datasets_asymmetree_medium_smaller),
    average_host_tree_depth(datasets_asymmetree_medium_normal),
]

# Compute average depths
parasite_depths = [
    average_parasite_tree_depth(datasets_asymmetree_medium_smaller),
    average_parasite_tree_depth(datasets_asymmetree_medium_normal),
    
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_depths, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_depths, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree Depth')
plt.title('Average Depth (# of branches) of Host and Parasite Trees per Asymmetree Model Medium')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

#Compute average depths
host_leaves = [
    average_host_tree_leaves(datasets_asymmetree_medium_smaller),
    average_host_tree_leaves(datasets_asymmetree_medium_normal),
]

# Compute average depths
parasite_leaves = [
    average_parasite_tree_leaves(datasets_asymmetree_medium_smaller),
    average_parasite_tree_leaves(datasets_asymmetree_medium_normal),
]

# Plotting
labels = ['Asymmetree 1.5', 'Asymmetree 2']
x = range(len(labels))

bar_width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - bar_width/2 for i in x], host_leaves, width=bar_width, label='Host Trees')
plt.bar([i + bar_width/2 for i in x], parasite_leaves, width=bar_width, label='Parasite Trees')
plt.xticks(ticks=x, labels=labels)
plt.ylabel('Average Tree #Leaves')
plt.title('Average #Leaves of Host and Parasite Trees per Asymmetree Model Medium')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import openpyxl
asymmetree_medium_small = compute_tree_metrics(datasets_asymmetree_medium_smaller, model_name="Asymmetree_Cosp_Small")
asymmetree_medium_normal = compute_tree_metrics(datasets_asymmetree_medium_normal, model_name="Asymmetree_Cosp_Normal")

comparison_metrics_medium = pd.concat([asymmetree_medium_small, asymmetree_medium_normal])
metrics_to_plot = [
    'cherry_index_host', 'cherry_index_parasite',
    'sackin_index_host', 'sackin_index_parasite',
    'sackin_index_topology_host', 'sackin_index_topology_parasite',
    'colless_index_host', 'colless_index_parasite',
]

plot_metric_comparison_paired(comparison_metrics_medium, 'cherry_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index')
plot_metric_comparison_paired(comparison_metrics_medium, 'sackin_index_topology')
plot_metric_comparison_paired(comparison_metrics_medium, 'colless_index')


In [ ]:
recon_medium_asymmetree_small = extract_sim_stats(datasets_asymmetree_medium_smaller)
recon_medium_asymmetree_norm = extract_sim_stats(datasets_asymmetree_medium_normal)

flattened_medium_asymmetree_small = flatten_sim_stats(recon_medium_asymmetree_small).fillna(0)
flattened_medium_asymmetree_norm = flatten_sim_stats(recon_medium_asymmetree_norm).fillna(0)

flattened_medium_asymmetree_small['Total Events'] = (flattened_medium_asymmetree_small['Horizontal_Gene_Transfer'] +
    flattened_medium_asymmetree_small['Duplication'] +
    flattened_medium_asymmetree_small['Speciation'] +
    flattened_medium_asymmetree_small['Loss'])
flattened_medium_asymmetree_small['Horizontal_Gene_Transfer_Rate'] = flattened_medium_asymmetree_small['Horizontal_Gene_Transfer'] / flattened_medium_asymmetree_small['Total Events']
flattened_medium_asymmetree_small['Duplication_Rate'] = flattened_medium_asymmetree_small['Duplication'] / flattened_medium_asymmetree_small['Total Events']
flattened_medium_asymmetree_small['Speciation_Rate'] = flattened_medium_asymmetree_small['Speciation'] / flattened_medium_asymmetree_small['Total Events']
flattened_medium_asymmetree_small['Loss_Rate'] = flattened_medium_asymmetree_small['Loss'] / flattened_medium_asymmetree_small['Total Events']
flattened_medium_asymmetree_small.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

flattened_medium_asymmetree_norm['Total Events'] = (flattened_medium_asymmetree_norm['Horizontal_Gene_Transfer'] +
    flattened_medium_asymmetree_norm['Duplication'] +
    flattened_medium_asymmetree_norm['Speciation'] +
    flattened_medium_asymmetree_norm['Loss'])
flattened_medium_asymmetree_norm['Horizontal_Gene_Transfer_Rate'] = flattened_medium_asymmetree_norm['Horizontal_Gene_Transfer'] / flattened_medium_asymmetree_norm['Total Events']
flattened_medium_asymmetree_norm['Duplication_Rate'] = flattened_medium_asymmetree_norm['Duplication'] / flattened_medium_asymmetree_norm['Total Events']
flattened_medium_asymmetree_norm['Speciation_Rate'] = flattened_medium_asymmetree_norm['Speciation'] / flattened_medium_asymmetree_norm['Total Events']
flattened_medium_asymmetree_norm['Loss_Rate'] = flattened_medium_asymmetree_norm['Loss'] / flattened_medium_asymmetree_norm['Total Events']
flattened_medium_asymmetree_norm.drop(columns=['Horizontal_Gene_Transfer', 'Duplication', 'Speciation', 'Loss'], inplace=True)

boxplot_asymmetree_medium_small = flattened_medium_asymmetree_small[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_medium_small.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)
boxplot_asymmetree_medium_norm = flattened_medium_asymmetree_norm[['filename', 'Speciation_Rate', 'Horizontal_Gene_Transfer_Rate',]]
boxplot_asymmetree_medium_norm.rename(columns={'Horizontal_Gene_Transfer_Rate': 'Host_switch', 'Speciation_Rate': 'Cospeciation'}, inplace=True)


boxplot_asymmetree_medium_small["Model"] = "AsymmeTree_small"
boxplot_asymmetree_medium_norm["Model"] = "AsymmeTree_norm"

# Combine all models into one DataFrame
df_box_medium = pd.concat(
    [boxplot_asymmetree_medium_small, boxplot_asymmetree_medium_norm],
    ignore_index=True
)

# Drop NaNs (optional, keeps only rows where values exist)
df_box_medium = df_box_medium.dropna(subset=["Cospeciation", "Host_switch"])
df_melted_medium = df_box_medium.melt(
    id_vars=["Model", "filename"],
    value_vars=["Cospeciation", "Host_switch"],
    var_name="Event",
    value_name="Frequency"
)
df_melted_medium


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_melted_medium,
    x="Model",
    y="Frequency",
    hue="Event",
    palette="Set2",
    width=0.6
)

plt.title("Cospeciation and Host-switch Frequencies Across Models Medium", fontsize=14, pad=15)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(title="Event", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()